# Campaign Targeting Decision Assistant
Marketing teams often need to decide which customers to contact when campaign capacity is limited. This assistant compares targeting strategies and explains which produces the most estimated incremental activations at a given contact volume.

This notebook records the development and evaluation history.
The saved outputs show the results of each experiment. Running all
cells repeats those experiments and appends new records to the test log.

A separate demonstration notebook will present the final v19 workflow.

Users can ask questions such as:
- Which strategy should I use when contacting 30% of customers?
- Does the recommendation change between 30% and 60%?
- How many additional activations does the recommended strategy produce compared with random targeting?

This notebook uses campaign evaluation results from a synthetic dataset and a previous analysis. The application combines existing campaign analysis with a local language model. Python retrieves the results and calculates comparisons. The language model interprets the question and explains the recommendation.

## 1. Setup and scope
The project uses Python, pandas, and Ollama. Ollama must be running locally with the `qwen3:4b-instruct-2507-q4_K_M` model installed.

Two CSV exports from the campaign modeling project are required:
- uplift_summary.csv
- response_summary.csv

Set `DATA_FOLDER` to the folder containing both files. The default configuration looks for a folder named campaign_assistant_data in the kernel’s working directory.

Run the cells in order. Each question is processed independently. Answers and supporting tool results are saved automatically so they can be reviewed later.

NOTE: Reproducing the complete development history also requires
`qwen3.5:9b`, which is used for the model comparison and versions
10 and 11.

In [38]:
import json
from pathlib import Path
from datetime import datetime
from time import perf_counter

import pandas as pd
from ollama import Client
from IPython.display import display, Markdown

# Change this path if the two CSV exports are stored elsewhere.
DATA_FOLDER = Path("campaign_assistant_data")

# Save test runs here without overwriting earlier results.
LOG_FILE = Path("assistant_tests.jsonl")

# Connect to the language model running on this computer.
MODEL = "qwen3:4b-instruct-2507-q4_K_M"
client = Client(host="http://localhost:11434")

# Identify the instruction version used for each saved run.
PROMPT_VERSION = "v1"

# Load the results from the existing campaign analysis.
cumulative_summary = pd.read_csv(DATA_FOLDER / "uplift_summary.csv")
cumulative_response_summary = pd.read_csv(DATA_FOLDER / "response_summary.csv")

# Confirm that both files loaded.
print("Uplift:", cumulative_summary.shape)
print("Response:", cumulative_response_summary.shape)

Uplift: (10, 13)
Response: (10, 13)


## 2. How the strategies are compared

**Response targeting** selects customers with the highest predicted probability of activation.

**Uplift targeting** selects customers with the highest predicted benefit from receiving treatment.

**Random targeting** provides a benchmark for selecting customers without prioritizing either score. It estimates incremental activations from randomly selecting customers from the holdout audience. It’s calculated by multiplying the full-audience *`estimated incremental activation count`* by the targeting share. 

For response and uplift targeting, the selected audiences are evaluated using their treatment and control activation rates. The assistant retrieves these existing results rather than estimating campaign outcomes itself.

The comparison measures estimated incremental activations at the same targeting depth. For example, a 30% comparison evaluates each strategy when contacting 30% of the audience.

In [39]:
def compare_targeting_strategies(targeting_share):
    """Return strategy estimates and pairwise comparisons at one targeting depth."""

    # Only accept the targeting depths available in the exported analysis.
    available_depths = [i / 10 for i in range(1, 11)]

    if targeting_share not in available_depths:
        raise ValueError("Choose a targeting share from 0.1 to 1.0 in increments of 0.1.")

    # Retrieve the uplift and response results at the requested depth.
    uplift_row = cumulative_summary.loc[
        cumulative_summary["cumulative_targeting_share"].round(6) == targeting_share].iloc[0]

    response_row = cumulative_response_summary.loc[
        cumulative_response_summary["cumulative_targeting_share"].round(6) == targeting_share].iloc[0]

    # Combine the three strategies into one ranked table.
    results = pd.DataFrame({"strategy": ["uplift", "response", "random"],
                            "estimated_incremental_activations": [uplift_row["cumulative_incremental_activations"],
                                                                  response_row["cumulative_incremental_activations"],
                                                                  uplift_row["random_targeting_incremental_activations"]]})

    results = results.sort_values("estimated_incremental_activations", ascending=False).reset_index(drop=True)

    # Look up each strategy's estimate by name.
    activations = results.set_index("strategy")["estimated_incremental_activations"]

    # Calculate comparisons in Python so the model doesn't need to do arithmetic.
    comparisons = []

    # Create pairs of combinations for comparisons
    pairs = [("uplift", "random"), ("uplift", "response"),
             ("response", "random"), ("response", "uplift")]

    # For each pair calculate activation difference and percent change
    for strategy, baseline in pairs:
        estimate = float(activations[strategy])
        baseline_estimate = float(activations[baseline])
        
        difference = estimate - baseline_estimate

        # Percent improvement is meaningful here only with a positive baseline.
        percent_change = difference / baseline_estimate * 100 if baseline_estimate > 0 else None

        comparisons.append({"strategy": strategy,
                            "baseline_strategy": baseline,
                            "activation_difference": difference,
                            "percent_change": percent_change,
                            "targeting_share": float(targeting_share)})
        
    return results, comparisons

## 3. Make the analysis available to the assistant
The tool connects the language model to the comparison function. It returns the ranked estimates and calculated differences with explicit labels.

Comparisons are stored as ordinary Python dictionaries. This keeps unavailable percentages as None, which converts directly to null in JSON.

In [40]:
def compare_targeting_tool(targeting_share: float) -> dict:
    """Compare strategies at a targeting share from 0.1 to 1.0 in 0.1 increments."""
    results, comparisons = compare_targeting_strategies(targeting_share)

    return {"targeting_share": float(targeting_share),
            "metric": "estimated_incremental_activations",
            "results": results.to_dict("records"),
            "comparisons": comparisons}

In [4]:
# Inspect the analytical output before involving the language model.
strategy_results, strategy_comparisons = compare_targeting_strategies(0.3)

display(strategy_results.round(2))
display(pd.DataFrame(strategy_comparisons).round(2))

,strategy,estimated_incremental_activations
0,uplift,274.22
1,random,219.89
2,response,182.38


,strategy,baseline_strategy,activation_difference,percent_change,targeting_share
0,uplift,random,54.33,24.71,0.3
1,uplift,response,91.84,50.36,0.3
2,response,random,-37.52,-17.06,0.3
3,response,uplift,-91.84,-33.49,0.3


In [5]:
print(json.dumps(compare_targeting_tool(0.3), indent=2))

{
  "targeting_share": 0.3,
  "metric": "estimated_incremental_activations",
  "results": [
    {
      "strategy": "uplift",
      "estimated_incremental_activations": 274.2159673399923
    },
    {
      "strategy": "random",
      "estimated_incremental_activations": 219.8908997853674
    },
    {
      "strategy": "response",
      "estimated_incremental_activations": 182.3752182563992
    }
  ],
  "comparisons": [
    {
      "strategy": "uplift",
      "baseline_strategy": "random",
      "activation_difference": 54.325067554624894,
      "percent_change": 24.705464213230687,
      "targeting_share": 0.3
    },
    {
      "strategy": "uplift",
      "baseline_strategy": "response",
      "activation_difference": 91.8407490835931,
      "percent_change": 50.35812977313358,
      "targeting_share": 0.3
    },
    {
      "strategy": "response",
      "baseline_strategy": "random",
      "activation_difference": -37.5156815289682,
      "percent_change": -17.06104325626334,
      "

## 4. Define the assistant’s responsibilities
The assistant should answer the business question, support its recommendation with the calculated results, and ask for clarification when necessary.
Its numerical evidence must come from the comparison tool. Questions about profit require additional cost and customer-value information.

In [10]:
ASSISTANT_INSTRUCTIONS = """
You help marketing teams choose a targeting strategy at a specified contact capacity.

An activation is the customer's defined target action within 30 days.
Incremental activations measure the additional activations from treatment
compared with no treatment.

Use compare_targeting_tool before reporting numerical campaign results.
Never claim tool use unless a tool result appears in the conversation.
Do not invent estimates or perform your own comparison arithmetic.

Convert percentages to fractions when requesting the tool.
Supported targeting shares are 0.1 through 1.0 in increments of 0.1.
You may request up to three tool calls, one for each requested depth.
If the depth is missing, ask what percentage of customers can be contacted.
Do not round unsupported depths or interpolate results.
Explain tool errors using the stated limitation.

Recommend the strategy with the highest estimated incremental activations.
Use the supplied comparisons to explain its advantage.
Name the comparison strategy and distinguish activation counts from percentages.
If percent_change is null, report only the activation difference.

For multiple depths, provide a compact table followed by a brief recommendation.
Round displayed estimates and percentages to two decimal places.
Keep answers under 150 words and limit conclusions to the supplied results.

Profit cannot be determined without costs and value per activation.
For profit questions, explain what is missing rather than substituting an
activation recommendation unless the user requests that alternative.
"""

# Preserve the original instructions for repeatable version comparisons.
V1_INSTRUCTIONS = ASSISTANT_INSTRUCTIONS

## 5. Ask a question and save the evidence
The first model request can answer directly, ask for clarification, or request the comparison tool.

When a tool is requested, Python executes the approved function and returns its results. A second model request explains the findings.

Each run saves the question, instructions, answer, tool activity, token usage, and elapsed time. Saving the instructions with the run preserves the test conditions even after the instructions are edited.

In [ ]:
def ask_assistant(question, test_name="exploration"):
    """Answer a question and automatically save the run for review."""

    if not isinstance(question, str) or not question.strip():
        raise ValueError("Enter a non-empty question.")

    started = perf_counter()
    responses = []
    tool_log = []

    messages = [{"role": "system", 
                 "content": ASSISTANT_INSTRUCTIONS},
                {"role": "user", 
                 "content": question}]

    # Let the model answer directly or request analytical results.
    first = client.chat(model=MODEL, 
                        messages=messages,
                        tools=[compare_targeting_tool],
                        stream=False, 
                        think=False,
                        options={"num_predict": 500, "temperature": 0})
    
    responses.append(first)
    calls = first.message.tool_calls or []

    if first.done_reason == "length":
        answer = "The request reached its output limit. Please narrow the question."

    elif len(calls) > 3:
        answer = "Please request at most three targeting depths."

    elif not calls:
        answer = first.message.content

    else:
        # Preserve the tool requests before adding their results.
        messages.append(first.message)

        for call in calls:
            name = call.function.name
            arguments = call.function.arguments

            try:
                # Execute only the approved function with its expected input.
                if name != "compare_targeting_tool":
                    raise ValueError("This tool is not available.")

                if set(arguments) != {"targeting_share"}:
                    raise ValueError("Supply only targeting_share.")

                tool_result = compare_targeting_tool(**arguments)

            except (ValueError, TypeError) as error:
                tool_result = {"status": "error", 
                               "message": str(error)}

            tool_log.append({"tool": name, 
                             "arguments": arguments, 
                             "result": tool_result})

            messages.append({"role": "tool", 
                             "tool_name": name,                
                             "content": json.dumps(tool_result)})

        # Collect error messages from failed tool calls.
        errors = []
        for item in tool_log:
            if item["result"].get("status") == "error":
                errors.append(item["result"]["message"])
    
        # If every call failed, show the errors directly.
        if len(errors) == len(tool_log):
            answer = " ".join(errors)
    
        else:
            # Ask the model to explain the results.
            final = client.chat(model=MODEL, 
                                messages=messages, 
                                stream=False,
                                think=False,
                                options={"num_predict": 500, "temperature": 0})
            responses.append(final)
            answer = final.message.content
    
            # Flag an incomplete response.
            if final.done_reason == "length":
                answer += "\n[Response reached the output limit.]"

    # Capture the evidence needed to assess this answer.
    run = {"timestamp": datetime.now().astimezone().isoformat(),
           "test_name": test_name,
           "prompt_version": PROMPT_VERSION,
           "instructions": ASSISTANT_INSTRUCTIONS,
           "model": MODEL,
           "question": question,
           "answer": answer,
           "tool_log": tool_log,
           "model_calls": len(responses),
           "input_tokens": sum(response.prompt_eval_count or 0 for response in responses),
           "output_tokens": sum(response.eval_count or 0 for response in responses),
           "stop_reasons": [response.done_reason for response in responses],
           "elapsed_seconds": round(perf_counter() - started, 2)}

    # Append the run to disk so later questions cannot overwrite it.
    with LOG_FILE.open("a", encoding="utf-8") as file:
        file.write(json.dumps(run, ensure_ascii=False, allow_nan=False) + "\n")

    return run

## 6. Evaluate the answers

A useful answer must do more than identify the winning strategy. It must use the correct values, explain the relevant comparison, and address the question asked.

| Test | Expected behavior |
|---|---|
| One supported depth | Correct strategy, estimates, and comparisons |
| Multiple supported depths | Correct recommendation at every requested depth |
| Unsupported depth | Explain the available choices without inventing results |
| Missing depth | Ask what percentage of customers to target  |
| Profit question | Explain the missing cost and value information |

Review the saved tool results alongside each answer. A direct answer may be appropriate for clarification or scope questions. Numerical campaign recommendations require a successful tool result.

In [7]:
# Change the question to explore another supported targeting decision.
result = ask_assistant("Compare 30% and 60% targeting. Recommend a strategy at each depth "
                       "and explain its advantage over the alternatives.",
                       test_name="multiple_depths")

In [8]:
# Inspect the saved results
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 30% depth, **uplift** has the highest incremental activations (274.22) and outperforms both random (219.90) and response (182.38) strategies by 54.33 and 91.84 activations, respectively.  

At 60% depth, **response** leads with 478.97 activations, surpassing random (439.78) by 39.19 and uplift (450.45) by 28.52.  

**Recommendation**:  
- At 30%: Use **uplift**—it delivers the largest gains over alternatives.  
- At 60%: Use **response**—it maximizes activation over random and uplift.  

The advantage lies in response’s superior performance at higher reach, while uplift excels at lower targeting with greater relative improvement.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.2159673399923
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.8908997853674
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.3752182563992
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.325067554624894,
          "percent_change": 24.705464213230687,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.8407490835931,
          "percent_change": 

In [7]:
TEST_QUESTIONS = {"single_depth": "Which strategy should I use at 30% targeting?",
                  "multiple_depths": "Compare 20%, 30%, and 60% targeting and explain the recommendations.",
                  "unsupported_depth": "Which strategy should I use at 35% targeting?",
                  "missing_depth": "Which targeting strategy should I use?",
                  "profit": "At 30% targeting, which strategy makes the most profit?"}

In [9]:
# Run one test at a time so its answer and tool evidence can be reviewed.
test_name = "profit"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [10]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

Profit cannot be determined without knowing the cost per contact and the value per activation. Please provide these details or clarify if you'd like a recommendation based on incremental activations only.

[]


In [11]:
test_name = "single_depth"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [12]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 30% targeting, use the **uplift** strategy. It generates the highest estimated incremental activations (274.22) compared to random (220.00) and response (182.38). Uplift outperforms random by 54.33 activations (24.71% increase) and response by 91.84 activations (50.36% increase). Response underperforms both, with a negative activation difference versus uplift. No profit can be determined without cost and value per activation.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.2159673399923
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.8908997853674
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.3752182563992
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.325067554624894,
          "percent_change": 24.705464213230687,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.8407490835931,
          "percent_change": 

In [13]:
test_name = "multiple_depths"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [14]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 20% targeting, "uplift" has the highest incremental activations (188.30) vs. "random" (+41.70) and "response" (+77.94). At 30%, "uplift" leads with 274.22 activations, outperforming "response" by 91.84. At 60%, "response" achieves 478.97 activations—highest of all—surpassing "uplift" by 28.52.  

**Recommendation**: For 60% targeting, use "response" strategy (478.97 activations), as it maximizes incremental activations. At lower shares (20% and 30%), "uplift" is optimal. The strategy shifts from "uplift" to "response" as targeting increases, due to higher engagement at greater reach.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.2
    },
    "result": {
      "targeting_share": 0.2,
      "metric": "estimated_incremental_activations",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 188.2912845866717
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 146.59393319024494
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 110.3536541010575
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 41.69735139642677,
          "percent_change": 28.44411804021472,
          "targeting_share": 0.2
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 77.9376304856142,
          "percent_change": 7

In [15]:
test_name = "unsupported_depth"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [16]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

Error: Targeting share must be between 0.1 and 1.0 in increments of 0.1. 35% (0.35) is valid, but the tool requires a value in that range. Please confirm the targeting share is exactly 0.35 or adjust if needed.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.35
    },
    "result": {
      "status": "error",
      "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
    }
  }
]


In [17]:
test_name = "missing_depth"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [18]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

Please specify the contact capacity (as a percentage from 0.1 to 1.0 in increments of 0.1) and the desired activation (e.g., conversion within 30 days) to determine the optimal targeting strategy. Without these details, I cannot recommend a strategy or perform the necessary comparisons.

[]


## 7. Record findings and improvements for prompt version v1

Each run saves the question, answer, instructions, and tool results. The table below records the manual review.

- **Pass** — Meets the test expectations.
- **Partial** — Provides a useful answer but contains issues.
- **Fail** — Gives incorrect results, invents information, or fails to address the question.

| Run | Test | Instruction version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 1 | Compare 30% and 60% | v1 | Correct winners and differences. Reported 219.90 instead of 219.89 and generalized beyond the evaluated depths. | Partial | Tighten numerical reporting and interpretation. |
| 2 | Profit question | v1 | Explained the missing cost and value information without inventing results. No tool call was needed. | Pass | Repeat after instruction changes. |
| 3 | Single depth, 30% | v1 | Correct winner, differences, and percentages. Reported random as 220.00 instead of 219.89 and added an unrequested profit statement. | Partial | Preserve numerical precision and answer only the question asked. |
| 4 | Compare 20%, 30%, and 60% | v1 | Correct winners and differences. Reported 188.30 instead of 188.29, omitted the requested table, and invented an explanation involving higher engagement. | Partial | Require the requested format and remove unsupported explanations. |
| 5 | Unsupported depth, 35% | v1 | The tool correctly rejected 35%, but the assistant incorrectly described it as valid. | Fail | Preserve the tool's stated limitation. |
| 6 | Missing depth | v1 | Asked for a depth but confused fractions with percentages and requested an activation definition already supplied. | Partial | Ask only what percentage of customers to target. |

Run numbers in the review tables are manual identifiers. The JSONL log
also contains additional executions, so a table's run number is not
necessarily the same as a line number in the log. Use the version,
test name, and timestamp to locate a recorded execution.

## 8. Amend Instructions for prompt version v2
The main finding in v1 is that the analytical tool is returning the correct results, but the model explanations remain inconsistent.  We will add new instructions to the assistant to further dial in these results and test in v2.

Version 2 also returns tool errors directly when every requested comparison fails. This prevents the model from contradicting the tool’s rejection of an unsupported targeting depth.

In [11]:
# Build v2 from the original instructions 
PROMPT_VERSION = "v2"

ASSISTANT_INSTRUCTIONS = V1_INSTRUCTIONS + """
Supported percentages are 10%, 20%, 30%, 40%, 50%, 60%, 70%, 80%, 90%, and 100%.
The corresponding tool inputs are 0.1 through 1.0 in increments of 0.1.
A value such as 35% is unsupported even though it lies between 10% and 100%.

If the depth is missing, ask only what percentage of customers to target.
The activation definition is already provided; do not ask the user to define it.

Report supplied numerical values rounded directly to two decimal places.
Do not round to whole numbers and then add decimal places.

For multiple depths, use a compact table followed by recommendations
specific to those depths.
Do not explain differences using engagement, customer behavior, or other
causes that are not supplied in the tool results.
Do not infer a general pattern across higher or lower targeting depths.

Mention profit limitations only when the question concerns profit.
"""
# Preserve the original instructions for repeatable version comparisons.
V2_INSTRUCTIONS = ASSISTANT_INSTRUCTIONS

In [27]:
test_name = "unsupported_depth"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [28]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

The requested targeting share of 35% is unsupported as it does not align with the supported increments of 0.1 (10%, 20%, ..., 100%). Please specify a supported percentage (e.g., 30%, 40%) or clarify your intent.

[]


In [29]:
test_name = "single_depth"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [30]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 30% targeting, use the **uplift** strategy.  

It generates the highest estimated incremental activations (274.22) compared to random (220.00) and response (182.38).  

Uplift outperforms random by 54.33 activations (24.71% increase) and response by 91.84 activations (50.36% increase).  

Response underperforms both, with a negative activation difference versus random and uplift.  

No profit analysis is possible without cost and value per activation.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.2159673399923
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.8908997853674
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.3752182563992
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.325067554624894,
          "percent_change": 24.705464213230687,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.8407490835931,
          "percent_change": 

In [31]:
print(result["prompt_version"])
print(result["instructions"])

v2

You help marketing teams choose a targeting strategy at a specified contact capacity.

An activation is the customer's defined target action within 30 days.
Incremental activations measure the additional activations from treatment
compared with no treatment.

Use compare_targeting_tool before reporting numerical campaign results.
Never claim tool use unless a tool result appears in the conversation.
Do not invent estimates or perform your own comparison arithmetic.

Convert percentages to fractions when requesting the tool.
Supported targeting shares are 0.1 through 1.0 in increments of 0.1.
You may request up to three tool calls, one for each requested depth.
If the depth is missing, ask what percentage of customers can be contacted.
Do not round unsupported depths or interpolate results.
Explain tool errors using the stated limitation.

Recommend the strategy with the highest estimated incremental activations.
Use the supplied comparisons to explain its advantage.
Name the compar

In [32]:
test_name = "missing_depth"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [33]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

To determine the best targeting strategy, I need to know the percentage of customers you'd like to contact (e.g., 10%, 20%, 30%, etc.). Please specify the targeting depth (as a percentage or fraction between 0.1 and 1.0 in 0.1 increments). Once provided, I’ll use the compare_targeting_tool to evaluate incremental activations and recommend the optimal strategy.

[]


In [34]:
test_name = "multiple_depths"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [35]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

| Targeting Share | Uplift (Activations) | Response (Activations) | Random (Activations) |
|----------------|------------------------|-------------------------|------------------------|
| 20%           | 188.29                | 110.35                 | 146.59                |
| 30%           | 274.22                | 182.38                 | 219.90                |
| 60%           | 450.45                | 478.97                 | 439.78                |

At 20% and 30%, "uplift" outperforms "response" and "random" by 28.4% and 50.4%, respectively. At 60%, "response" achieves 478.97 activations—highest of all—surpassing "uplift" by 28.5. Thus, for 20% and 30%, prioritize "uplift"; for 60%, choose "response" for maximum incremental activations.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.2
    },
    "result": {
      "targeting_share": 0.2,
      "metric": "estimated_incremental_activations",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 188.2912845866717
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 146.59393319024494
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 110.3536541010575
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 41.69735139642677,
          "percent_change": 28.44411804021472,
          "targeting_share": 0.2
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 77.9376304856142,
          "percent_change": 7

In [36]:
test_name = "profit"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [37]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 30% targeting (0.3), profit cannot be determined without knowing the cost per activation and the value per activation. The tool provides activation counts and incremental activations but does not include financial metrics. Please provide cost and value per activation to assess profit.

[]


In [39]:
result = ask_assistant("Compare 30% and 60% targeting. Recommend a strategy at each depth "
                       "and explain its advantage over the alternatives.",
                       test_name="two_depths")

In [40]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 30% targeting:  
- **Uplift** has the highest incremental activations (274.22) and outperforms random (219.89) by 54.33, a 24.71% increase. It also exceeds response (182.38) by 91.84, a 50.36% gain.  
- **Response** underperforms both, especially vs. uplift.  

At 60% targeting:  
- **Response** leads with 478.97 activations, 39.19 more than random (439.78), a 8.91% increase.  
- **Uplift** trails response by 28.52, a 5.95% drop.  

**Recommendation**:  
- At 30%: Use **uplift** for highest incremental activations.  
- At 60%: Use **response** for superior activation gains despite higher targeting.  

Both strategies outperform random; uplift dominates at lower depth, response at higher.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.2159673399923
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.8908997853674
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.3752182563992
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.325067554624894,
          "percent_change": 24.705464213230687,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.8407490835931,
          "percent_change": 

## 9. Record findings and improvements for prompt version v2

Each run saves the question, answer, instructions, and tool results. The table below records the manual review.

- **Pass** — Meets the test expectations.
- **Partial** — Provides a useful answer but contains issues.
- **Fail** — Gives incorrect results, invents information, or fails to address the question.

| Run | Test | Instruction version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 7 | Unsupported depth, 35% | v2 | Correctly explained that 35% is unsupported and requested an available depth. No tool call was made. | Pass | Direct-answer behavior verified; tool-error handling not exercised. |
| 8 | Single depth, 30% | v2 | Correct winner, differences, and percentages. Still reported random as 220.00 instead of 219.89 and included an unrequested profit statement. | Partial | Numerical precision and answer relevance remain unresolved. |
| 9 | Missing depth | v2 | Correctly requested the targeting depth without requesting an activation definition. Still mixed percentage and fraction units when describing supported inputs. | Partial | Clarify how supported depths are communicated. |
| 10 | Compare 20%, 30%, and 60% | v2 | Correct tool calls, winners, and table structure. Assigned percentage advantages to the wrong comparisons and reported 219.90 instead of 219.89. | Fail | Keep each comparison explicitly tied to its targeting depth and comparison strategy. |
| 11 | Profit question | v2 | Correctly explained that profit cannot be determined without cost and value information. Provided no invented results and made no unnecessary tool call. | Pass | No change needed for this test. |
| 12 | Compare 30% and 60% | v2 | All reported numbers and recommendations were correct. Incorrectly stated that both strategies outperform random, added unsupported interpretation, and omitted the table. | Partial | Preserve accurate comparisons and remove claims contradicted by the results. |

## 10. Simplify the tool output for version 3

Versions 1 and 2 supplied all four comparisons at full precision. The assistant sometimes mixed up comparison percentages or reported values incorrectly.

Version 3 keeps the version 2 instructions unchanged. It changes the tool output by rounding numerical results to two decimal places, explicitly naming the leading strategy, and including only that strategy's comparisons against the alternatives.

The same questions are repeated to evaluate whether this change improves answer accuracy.

In [42]:
# Redefine the tool for v3. Subsequent tests use this version.
def compare_targeting_tool(targeting_share: float) -> dict:
    """Compare strategies at a targeting share from 0.1 to 1.0 in 0.1 increments."""
    results, comparisons = compare_targeting_strategies(targeting_share)

    # Results are sorted from highest to lowest.
    leader = results.iloc[0]["strategy"]
    result_records = results.round(2).to_dict("records")

    # Send only the leading strategy's comparisons.
    leader_comparisons = []
    for comparison in comparisons:
        if comparison["strategy"] == leader:
            row = comparison.copy()
            row["activation_difference"] = round(row["activation_difference"], 2)

            if row["percent_change"] is not None:
                row["percent_change"] = round(row["percent_change"], 2)

            leader_comparisons.append(row)

    return {"targeting_share": float(targeting_share),
            "metric": "estimated_incremental_activations",
            "leading_strategy": leader,
            "results": result_records,
            "comparisons": leader_comparisons}


# Keep the v2 instructions and label subsequent runs as v3.
ASSISTANT_INSTRUCTIONS = V2_INSTRUCTIONS
PROMPT_VERSION = "v3"

In [75]:
test_name = "unsupported_depth"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [76]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

The requested targeting share of 35% (0.35) is unsupported as it does not align with the allowed increments of 0.1 from 0.1 to 1.0. Please specify a supported percentage (e.g., 10%, 20%, 30%, 40%, 50%, 60%, 70%, 80%, 90%, or 100%).

[]


In [65]:
test_name = "multiple_depths"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [66]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

| Targeting Share | Leading Strategy | Estimated Incremental Activations |
|----------------|------------------|-----------------------------------|
| 20%           | Uplift           | 188.29                           |
| 30%           | Uplift           | 274.22                           |
| 60%           | Response         | 478.97                           |

At 20% and 30% targeting, "uplift" delivers the highest incremental activations, outperforming "random" and "response" by 28.4% and 24.7% respectively. At 60%, "response" becomes optimal, with 39.19 more activations than "random" (8.91% increase), though it lags "uplift" by 28.52 activations (6.33% less).  

Recommendation: For 20%–30% targeting, use "uplift" for maximum incremental activations. At 60%, shift to "response" for higher activation volume, despite a smaller relative gain. Profit cannot be determined without cost and value per activation.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.2
    },
    "result": {
      "targeting_share": 0.2,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 188.29
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 146.59
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 110.35
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 41.7,
          "percent_change": 28.44,
          "targeting_share": 0.2
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 77.94,
          "percent_change": 70.63,
          "targeting_share":

In [67]:
test_name = "missing_depth"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [68]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

To determine the best targeting strategy, I need to know the percentage of customers you'd like to contact (e.g., 10%, 20%, ..., 100%). Please specify the targeting depth (as a percentage or fraction between 0.1 and 1.0 in 0.1 increments). Once provided, I’ll use the compare_targeting_tool to evaluate incremental activations and recommend the optimal strategy.

[]


In [69]:
test_name = "single_depth"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [70]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 30% targeting, use the **uplift** strategy.  

It generates the highest estimated incremental activations (274.22) compared to random (219.89) and response (182.38).  

- Uplift vs. random: 54.33 more activations (24.71% increase).  
- Uplift vs. response: 91.84 more activations (50.36% increase).  

The uplift strategy outperforms both alternatives by a significant margin at this depth.  

Profit cannot be determined without cost and value per activation.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.84,
          "percent_change": 50.36,
          "targeting_share"

In [71]:
test_name = "profit"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [72]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 30% targeting (0.3), the strategy with the highest estimated incremental activations is required to determine profit. However, profit cannot be determined without knowing the cost per activation and the value per activation.  

Please confirm if you would like me to recommend the targeting strategy based on incremental activations, or if you have additional details about activation value and cost.

[]


In [73]:
result = ask_assistant("Compare 30% and 60% targeting. Recommend a strategy at each depth "
                       "and explain its advantage over the alternatives.",
                       test_name="two_depths")

In [74]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 30% targeting, use "uplift" strategy. It generates 274.22 incremental activations—54.33 more than "random" (24.71% increase) and 91.84 more than "response" (50.36% increase), outperforming alternatives.

At 60% targeting, use "response" strategy. It delivers 478.97 incremental activations—39.19 more than "random" (8.91% increase) and 28.52 more than "uplift" (6.33% increase), showing the highest advantage over "uplift".

Both strategies maximize incremental activations at their respective depths.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.84,
          "percent_change": 50.36,
          "targeting_share"

## 11. Record findings and improvements for prompt version v3

Each run saves the question, answer, instructions, and tool results. The table below records the manual review.

- **Pass** — Meets the test expectations.
- **Partial** — Provides a useful answer but contains issues.
- **Fail** — Gives incorrect results, invents information, or fails to address the question.

| Run | Test | Instruction version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 13 | Unsupported depth, 35% | v3 | Correctly rejected 35% and listed supported percentages. No tool call was needed. | Pass | No change needed. |
| 14 | Compare 20%, 30%, and 60% | v3 | Correct winners and table values. Mixed percentages between comparisons and incorrectly said response trails uplift at 60%, reversing the actual result. | Fail | Address comparison direction and separation between depths. |
| 15 | Missing depth | v3 | Requested the targeting depth but still described percentages and fractions using the same numerical range. | Partial | Use percentage units consistently when asking for input. |
| 16 | Single depth, 30% | v3 | Correct estimates, differences, percentages, and recommendation. Still included an unrequested profit statement. | Partial | Keep the answer focused on the question. |
| 17 | Profit question | v3 | Correctly declined to determine profit without financial inputs, but incorrectly suggested that identifying the activation leader is required to determine profit. | Partial | Remove the unsupported prerequisite. |
| 18 | Compare 30% and 60% | v3 | Correct estimates, differences, percentages, and recommendations. Omitted the table and incorrectly described the 60% advantage over uplift as the highest, although the advantage over random was larger. | Partial | Improve formatting compliance and comparison wording. |

### Findings

The revised tool output produced correctly rounded estimates and comparisons in the single-depth test. However, the three-depth answer still mixed percentages between comparisons and reversed the direction of one advantage.

These results show that simplifying the tool output helped numerical reporting in one example but did not consistently resolve interpretation errors across multiple targeting depths.

## 12. Focus recommendations on each targeting depth

Version 3 returned correct analytical results, but the assistant still combined comparisons from different depths and reversed one comparison.

Version 4 keeps the tool unchanged and replaces the accumulated instructions with a shorter set. Each recommendation must identify its targeting depth, leading strategy, and advantage over a named alternative.

The same questions will be repeated to evaluate the change.

In [13]:
PROMPT_VERSION = "v4"

# Replace accumulated instructions with a shorter, explicit response structure.
ASSISTANT_INSTRUCTIONS = """
You help marketing teams compare targeting strategies using estimated
incremental activations.

Use compare_targeting_tool for numerical campaign questions.
Request one tool call per depth, with a maximum of three.
Supported percentages are 10% through 100% in steps of 10%.
Convert these to fractions for the tool: 30% means 0.3.
Never round an unsupported depth to an available one.

If the depth is missing, ask only:
"What percentage of customers would you like to target?"

For each successful tool result, write a separate recommendation:
"At [percentage] targeting, choose [leading_strategy], with [estimate]
estimated incremental activations."
Then describe its supplied comparisons, naming each alternative.
Keep each comparison attached to its own depth.
Copy supplied numbers without recalculating or changing their direction.
Do not combine comparisons from different depths into one sentence.

Explain only the supplied results. Do not add causes or broader trends.
Discuss profit only when asked. Profit requires cost and value information.
Do not report numerical campaign results without a successful tool result.
Keep the answer concise.
"""
# Preserve the instructions used for the v4 tests.
V4_INSTRUCTIONS = ASSISTANT_INSTRUCTIONS

In [13]:
test_name = "multiple_depths"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [14]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 20% targeting, choose uplift, with 188.29 estimated incremental activations.  
Compared to random, uplift generates 41.7 more activations (a 28.44% increase). Compared to response, it generates 77.94 more activations (a 70.63% increase).

At 30% targeting, choose uplift, with 274.22 estimated incremental activations.  
Compared to random, uplift generates 54.33 more activations (a 24.71% increase). Compared to response, it generates 91.84 more activations (a 50.36% increase).

At 60% targeting, choose response, with 478.97 estimated incremental activations.  
Compared to random, response generates 39.19 more activations (an 8.91% increase). Compared to uplift, it generates 28.52 more activations (a 6.33% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.2
    },
    "result": {
      "targeting_share": 0.2,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 188.29
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 146.59
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 110.35
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 41.7,
          "percent_change": 28.44,
          "targeting_share": 0.2
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 77.94,
          "percent_change": 70.63,
          "targeting_share":

In [15]:
result = ask_assistant("Compare 30% and 60% targeting. Recommend a strategy at each depth "
                       "and explain its advantage over the alternatives.",
                       test_name="two_depths")

In [16]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 30% targeting, choose uplift, with 274.22 estimated incremental activations.  
Uplift outperforms random by 54.33 activations (a 24.71% increase) and outperforms response by 91.84 activations (a 50.36% increase).

At 60% targeting, choose response, with 478.97 estimated incremental activations.  
Response outperforms random by 39.19 activations (an 8.91% increase) and outperforms uplift by 28.52 activations (a 6.33% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.84,
          "percent_change": 50.36,
          "targeting_share"

In [17]:
test_name = "missing_depth"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [18]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

What percentage of customers would you like to target?

[]


In [19]:
test_name = "single_depth"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [20]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 30% targeting, choose uplift, with 274.22 estimated incremental activations.

Compared to random, uplift generates 54.33 more estimated incremental activations, a 24.71% increase at 30% targeting.  
Compared to response, uplift generates 91.84 more estimated incremental activations, a 50.36% increase at 30% targeting.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.84,
          "percent_change": 50.36,
          "targeting_share"

In [21]:
test_name = "unsupported_depth"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [22]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

Choose a targeting share from 0.1 to 1.0 in increments of 0.1.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.35
    },
    "result": {
      "status": "error",
      "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
    }
  }
]


In [23]:
test_name = "profit"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [24]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 30% targeting, choose uplift, with 274.22 estimated incremental activations.  
Compared to random, uplift generates 54.33 more estimated incremental activations (a 24.71% increase).  
Compared to response, uplift generates 91.84 more estimated incremental activations (a 50.36% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.84,
          "percent_change": 50.36,
          "targeting_share"

## 13. Record findings and improvements for prompt version v4

Each run saves the question, answer, instructions, and tool results. The table below records the manual review.

- **Pass** — Meets the test expectations.
- **Partial** — Provides a useful answer but contains issues.
- **Fail** — Gives incorrect results, invents information, or fails to address the question.

| Run | Test | Instruction version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 19 | Compare 20%, 30%, and 60% | v4 | Correct winners, estimates, differences, and percentages. Kept comparisons attached to their respective depths without unsupported explanations. | Pass | Preserve this response structure. |
| 20 | Compare 30% and 60% | v4 | Correct recommendations and all supporting comparisons. No unsupported conclusions. | Pass | Preserve this response structure. |
| 21 | Missing depth | v4 | Asked only what percentage of customers to target. | Pass | No change needed. |
| 22 | Single depth, 30% | v4 | Correct recommendation, estimate, differences, and percentages. No unrequested profit discussion. | Pass | No change needed. |
| 23 | Unsupported depth, 35% | v4 | The tool rejected 35% and the application returned its error directly without model reinterpretation. | Pass | Tool-error handling verified. |
| 24 | Profit question | v4 | Returned an activation recommendation without explaining that profit could not be determined. | Fail | Restore an explicit instruction for profit questions. |

## 14. Clarify the response to profit questions

Version 4 correctly handled the targeting comparisons, missing depth, and unsupported depth tests. However, it answered the profit question with an activation recommendation instead of explaining that the financial information was unavailable.

Version 5 keeps the tool and comparison instructions unchanged. It strengthens the instruction to explain that profit requires cost and value information, without substituting an activation recommendation.

The profit question will be tested first, followed by the remaining questions to check that the successful version 4 behavior is preserved.

In [14]:
# Update prompt version for v5
PROMPT_VERSION = "v5"

# Keep successful comparison instructions, update profit
ASSISTANT_INSTRUCTIONS = """
You help marketing teams compare targeting strategies using estimated
incremental activations.

Use compare_targeting_tool for numerical campaign questions.
Request one tool call per depth, with a maximum of three.
Supported percentages are 10% through 100% in steps of 10%.
Convert these to fractions for the tool: 30% means 0.3.
Never round an unsupported depth to an available one.

If the depth is missing, ask only:
"What percentage of customers would you like to target?"

For each successful tool result, write a separate recommendation:
"At [percentage] targeting, choose [leading_strategy], with [estimate]
estimated incremental activations."
Then describe its supplied comparisons, naming each alternative.
Keep each comparison attached to its own depth.
Copy supplied numbers without recalculating or changing their direction.
Do not combine comparisons from different depths into one sentence.

Explain only the supplied results. Do not add causes or broader trends.
If the user asks which strategy produces the most profit, explain that
profit cannot be determined from the available activation results.
Cost and value information are required.
Do not substitute an activation recommendation for a profit answer.
Offer an activation comparison only if the user requests that alternative.

Do not report numerical campaign results without a successful tool result.
Keep the answer concise.
"""
# Preserve the instructions used for the v5 tests.
V5_INSTRUCTIONS = ASSISTANT_INSTRUCTIONS

In [28]:
test_name = "profit"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [29]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

Profit cannot be determined from the available activation results. Cost and value information are required to assess profit.

[]


In [30]:
test_name = "unsupported_depth"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [31]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

Choose a targeting share from 0.1 to 1.0 in increments of 0.1.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.35
    },
    "result": {
      "status": "error",
      "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
    }
  }
]


In [32]:
test_name = "single_depth"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [33]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 30% targeting, choose uplift, with 274.22 estimated incremental activations.

Compared to random, uplift generates 54.33 more estimated incremental activations, a 24.71% increase at 30% targeting.  
Compared to response, uplift generates 91.84 more estimated incremental activations, a 50.36% increase at 30% targeting.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.84,
          "percent_change": 50.36,
          "targeting_share"

In [34]:
test_name = "missing_depth"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [35]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

What percentage of customers would you like to target?

[]


In [36]:
result = ask_assistant("Compare 30% and 60% targeting. Recommend a strategy at each depth "
                       "and explain its advantage over the alternatives.",
                       test_name="two_depths")

In [37]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 30% targeting, choose uplift, with 274.22 estimated incremental activations.  
Uplift outperforms random by 54.33 activations (a 24.71% increase) and outperforms response by 91.84 activations (a 50.36% increase).

At 60% targeting, choose response, with 478.97 estimated incremental activations.  
Response outperforms random by 39.19 activations (an 8.91% increase) and outperforms uplift by 28.52 activations (a 6.33% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.84,
          "percent_change": 50.36,
          "targeting_share"

In [38]:
test_name = "multiple_depths"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [39]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 20% targeting, choose uplift, with 188.29 estimated incremental activations.  
Compared to random, uplift generates 41.7 more activations (a 28.44% increase). Compared to response, it generates 77.94 more activations (a 70.63% increase).

At 30% targeting, choose uplift, with 274.22 estimated incremental activations.  
Compared to random, uplift generates 54.33 more activations (a 24.71% increase). Compared to response, it generates 91.84 more activations (a 50.36% increase).

At 60% targeting, choose response, with 478.97 estimated incremental activations.  
Compared to random, response generates 39.19 more activations (an 8.91% increase). Compared to uplift, it generates 28.52 more activations (a 6.33% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.2
    },
    "result": {
      "targeting_share": 0.2,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 188.29
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 146.59
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 110.35
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 41.7,
          "percent_change": 28.44,
          "targeting_share": 0.2
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 77.94,
          "percent_change": 70.63,
          "targeting_share":

## 15. Record findings and improvements for prompt version v5

Each run saves the question, answer, instructions, and tool results. The table below records the manual review.

- **Pass** — Meets the test expectations.
- **Partial** — Provides a useful answer but contains issues.
- **Fail** — Gives incorrect results, invents information, or fails to address the question.

| Run | Test | Instruction version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 25 | Profit question | v5 | Explained that profit requires cost and value information without substituting an activation recommendation. | Pass | Preserve this behavior. |
| 26 | Unsupported depth, 35% | v5 | Returned the tool's rejection directly without incorrect reinterpretation. | Pass | Preserve this behavior. |
| 27 | Single depth, 30% | v5 | Correct recommendation, estimate, activation differences, and percentages. | Pass | Preserve this behavior. |
| 28 | Missing depth | v5 | Asked only what percentage of customers to target. | Pass | Preserve this behavior. |
| 29 | Compare 30% and 60% | v5 | Correct recommendations and comparisons, kept separate by depth. | Pass | Preserve this behavior. |
| 30 | Compare 20%, 30%, and 60% | v5 | Correct estimates, recommendations, differences, and percentages at all requested depths. | Pass | Preserve this behavior. |

## 16. Findings

Version 5 met the acceptance criteria for all six evaluation questions in this test round.

Python supplied the campaign estimates and calculated comparisons. Simplifying the tool output and using a separate recommendation for each targeting depth helped address the numerical and interpretation errors observed in earlier versions.

The evaluation also identified a regression in profit-question handling. A targeted instruction change restored that boundary while preserving accurate targeting comparisons.

These results support a working demonstration of the use case. Additional questions and targeting depths are needed to assess how well the behavior extends beyond the examples used during development.

## 17. Evaluate new questions

The version 5 instructions and tool remain unchanged for this evaluation.

These questions introduce different targeting depths and wording from the development tests. The objective is to check whether the assistant applies the same behavior beyond the examples used to refine it.

Answers will be reviewed for correct targeting shares, numerical accuracy, comparison direction, and relevance. Expected numerical results will be checked against the analytical function.

In [15]:
NEW_QUESTIONS = {"new_depth": "We can contact 40% of customers. Which strategy should we use and why?",
                 "new_depth_comparison": "Recommend a targeting strategy separately at 50% and 70%, including the advantage over alternatives.",
                 "fraction_input": "Our targeting share is 0.8. What do you recommend?",
                 "unsupported_wording": "We can reach one quarter of our customers. Which strategy should we use?",
                 "profit_reworded": "At 50% targeting, which approach will make us the most money?"}

In [41]:
test_name = "new_depth"
result = ask_assistant(NEW_QUESTIONS[test_name], test_name)

In [42]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 40% targeting, choose random, with 293.19 estimated incremental activations.  
- uplift: 286.43 estimated incremental activations  
- response: 258.69 estimated incremental activations

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.4
    },
    "result": {
      "targeting_share": 0.4,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "random",
      "results": [
        {
          "strategy": "random",
          "estimated_incremental_activations": 293.19
        },
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 286.43
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 258.69
        }
      ],
      "comparisons": []
    }
  }
]


In [43]:
# Check new depth - retrieve the reference results directly from the analytical function
reference_results, reference_comparisons = compare_targeting_strategies(0.4)
display(reference_results.round(2))
display(pd.DataFrame(reference_comparisons).round(2))

,strategy,estimated_incremental_activations
0,random,293.19
1,uplift,286.43
2,response,258.69


,strategy,baseline_strategy,activation_difference,percent_change,targeting_share
0,uplift,random,-6.76,-2.30,0.4
1,uplift,response,27.74,10.72,0.4
2,response,random,-34.50,-11.77,0.4
3,response,uplift,-27.74,-9.69,0.4


In [44]:
test_name = "new_depth_comparison"
result = ask_assistant(NEW_QUESTIONS[test_name], test_name)

In [45]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 50% targeting, choose uplift, with 381.7 estimated incremental activations.  
Compared to random, uplift has 15.22 more activations (a 4.15% increase).  
Compared to response, uplift has 12.82 more activations (a 3.48% increase).

At 70% targeting, choose response, with 562.19 estimated incremental activations.  
Compared to random, response has 49.12 more activations (a 9.57% increase).  
Compared to uplift, response has 57.42 more activations (an 11.37% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.5
    },
    "result": {
      "targeting_share": 0.5,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 381.7
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 368.88
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 366.48
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 15.22,
          "percent_change": 4.15,
          "targeting_share": 0.5
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 12.82,
          "percent_change": 3.48,
          "targeting_share": 0

In [51]:
# Check new depth comparision - retrieve the reference results directly from the analytical function
reference_results, reference_comparisons = compare_targeting_strategies(0.5)
display(reference_results.round(2))
display(pd.DataFrame(reference_comparisons).round(2))

reference_results, reference_comparisons = compare_targeting_strategies(0.7)
display(reference_results.round(2))
display(pd.DataFrame(reference_comparisons).round(2))

,strategy,estimated_incremental_activations
0,uplift,381.70
1,response,368.88
2,random,366.48


,strategy,baseline_strategy,activation_difference,percent_change,targeting_share
0,uplift,random,15.22,4.15,0.5
1,uplift,response,12.82,3.48,0.5
2,response,random,2.40,0.65,0.5
3,response,uplift,-12.82,-3.36,0.5


,strategy,estimated_incremental_activations
0,response,562.19
1,random,513.08
2,uplift,504.78


,strategy,baseline_strategy,activation_difference,percent_change,targeting_share
0,uplift,random,-8.30,-1.62,0.7
1,uplift,response,-57.42,-10.21,0.7
2,response,random,49.12,9.57,0.7
3,response,uplift,57.42,11.37,0.7


In [47]:
test_name = "fraction_input"
result = ask_assistant(NEW_QUESTIONS[test_name], test_name)

In [48]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 80% targeting, choose leading_strategy, with 150 estimated incremental activations.  
Comparisons:  
- At 80% targeting, alternative_strategy has 140 estimated incremental activations.  
- At 80% targeting, secondary_strategy has 135 estimated incremental activations.  
- At 80% targeting, tertiary_strategy has 120 estimated incremental activations.  

Note: Profit cannot be determined from activation results alone. Cost and value information are required to assess profitability.

[]


In [52]:
reference_results, reference_comparisons = compare_targeting_strategies(0.8)
display(reference_results.round(2))
display(pd.DataFrame(reference_comparisons).round(2))

,strategy,estimated_incremental_activations
0,uplift,610.66
1,response,591.62
2,random,586.38


,strategy,baseline_strategy,activation_difference,percent_change,targeting_share
0,uplift,random,24.29,4.14,0.8
1,uplift,response,19.05,3.22,0.8
2,response,random,5.24,0.89,0.8
3,response,uplift,-19.05,-3.12,0.8


In [49]:
test_name = "unsupported_wording"
result = ask_assistant(NEW_QUESTIONS[test_name], test_name)

In [50]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

"What percentage of customers would you like to target?"

[]


In [53]:
test_name = "profit_reworded"
result = ask_assistant(NEW_QUESTIONS[test_name], test_name)

In [54]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 50% targeting, choose [leading_strategy], with [estimate] estimated incremental activations.

The comparison includes:
- [alternative_1]: [value] estimated incremental activations
- [alternative_2]: [value] estimated incremental activations
- [alternative_3]: [value] estimated incremental activations

Note: Profit cannot be determined from the available activation results. Cost and value information are required to assess which strategy produces the most profit.

[]


## 18. Results for new questions

| Run | Test | Version | Observed behavior | Outcome |
|---|---|---|---|---|
| 31 | New depth, 40% | v5 | Correctly selected random and reported all strategy estimates. The tool returned no comparisons for the winning strategy, so the explanation lacked calculated advantages. | Partial |
| 32 | Compare 50% and 70% | v5 | Correct recommendations, estimates, differences, and percentages at both depths. | Pass |
| 33 | Fraction input, 0.8 | v5 | Made no tool call and invented estimates and strategy names. | Fail |
| 34 | One-quarter targeting | v5 | Asked for a percentage instead of recognizing one quarter as the unsupported depth of 25%. | Partial |
| 35 | Reworded profit question | v5 | Explained the missing financial inputs but also returned an unrequested activation template with unresolved placeholders. | Partial |

### Findings from new questions

The assistant answered the 50% and 70% comparison correctly, but performance did not transfer consistently to new wording.

Testing identified a missing comparison case when random targeting led, fabricated results for a fraction-based input, and incomplete handling of conversational expressions such as “one quarter” and “most money.”

Passing the development questions therefore did not establish reliable performance across the supported use case.

## 19. Add comparisons when random targeting leads

The 40% targeting test identified a gap in the analytical output. Random targeting had the highest estimated incremental activations, but the comparison function only calculated advantages for uplift and response. The assistant therefore received the correct ranking without comparisons explaining random targeting's advantage.

Version 6 adds random-versus-uplift and random-versus-response comparisons. The revised function is defined below so earlier sections retain their original calculation logic when the notebook is run in order.

The instructions and tool wrapper remain unchanged to evaluate this calculation change separately.

In [43]:
# Update prompt version for v6
PROMPT_VERSION = "v6"

# Update pairs to include random vs uplist and random vs response
def compare_targeting_strategies(targeting_share):
    """Return strategy estimates and pairwise comparisons at one targeting depth."""

    # Only accept the targeting depths available in the exported analysis.
    available_depths = [i / 10 for i in range(1, 11)]

    if targeting_share not in available_depths:
        raise ValueError("Choose a targeting share from 0.1 to 1.0 in increments of 0.1.")

    # Retrieve the uplift and response results at the requested depth.
    uplift_row = cumulative_summary.loc[
        cumulative_summary["cumulative_targeting_share"].round(6) == targeting_share].iloc[0]

    response_row = cumulative_response_summary.loc[
        cumulative_response_summary["cumulative_targeting_share"].round(6) == targeting_share].iloc[0]

    # Combine the three strategies into one ranked table.
    results = pd.DataFrame({"strategy": ["uplift", "response", "random"],
                            "estimated_incremental_activations": [uplift_row["cumulative_incremental_activations"],
                                                                  response_row["cumulative_incremental_activations"],
                                                                  uplift_row["random_targeting_incremental_activations"]]})

    results = results.sort_values("estimated_incremental_activations", ascending=False).reset_index(drop=True)

    # Look up each strategy's estimate by name.
    activations = results.set_index("strategy")["estimated_incremental_activations"]

    # Calculate comparisons in Python so the model doesn't need to do arithmetic.
    comparisons = []

    # Create pairs of combinations for comparisons
    pairs = [("uplift", "random"),
             ("uplift", "response"),
             ("response", "random"), 
             ("response", "uplift"),
             ("random", "uplift"), 
             ("random", "response")]

    # For each pair calculate activation difference and percent change
    for strategy, baseline in pairs:
        estimate = float(activations[strategy])
        baseline_estimate = float(activations[baseline])
        
        difference = estimate - baseline_estimate

        # Percent improvement is meaningful here only with a positive baseline.
        percent_change = difference / baseline_estimate * 100 if baseline_estimate > 0 else None

        comparisons.append({"strategy": strategy,
                            "baseline_strategy": baseline,
                            "activation_difference": difference,
                            "percent_change": percent_change,
                            "targeting_share": float(targeting_share)})
        
    return results, comparisons

### Retest random targeting at 40%

Repeat the original 40% question and inspect the tool output.

The expected result is random targeting as the leading strategy, with comparisons against both uplift and response. Check that the assistant reports the supplied differences and percentages with the correct comparison direction.

This test evaluates the missing-comparison fix. The fabricated answer observed for fraction-based input remains a separate unresolved issue.

In [58]:
test_name = "new_depth"
result = ask_assistant(NEW_QUESTIONS[test_name], test_name)

In [59]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 40% targeting, choose random, with 293.19 estimated incremental activations.

Compared to uplift, random has 6.76 more estimated incremental activations, representing a 2.36% increase.  
Compared to response, random has 34.5 more estimated incremental activations, representing a 13.34% increase.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.4
    },
    "result": {
      "targeting_share": 0.4,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "random",
      "results": [
        {
          "strategy": "random",
          "estimated_incremental_activations": 293.19
        },
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 286.43
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 258.69
        }
      ],
      "comparisons": [
        {
          "strategy": "random",
          "baseline_strategy": "uplift",
          "activation_difference": 6.76,
          "percent_change": 2.36,
          "targeting_share": 0.4
        },
        {
          "strategy": "random",
          "baseline_strategy": "response",
          "activation_difference": 34.5,
          "percent_change": 13.34,
          "targeting_share": 0

## 20. Record findings for version 6

Review the answer against the analytical results and record the outcome below.

| Run | Test | Version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 36 | Random targeting leads at 40% | v6 | Correctly selected random and reported both calculated advantages with accurate differences, percentages, and comparison direction. | Pass | Missing-comparison fix verified for this case. |

In [64]:
# Preserve the instructions used for the v6 test.
V6_INSTRUCTIONS = ASSISTANT_INSTRUCTIONS

## 21. Recognize fraction-based targeting inputs

An earlier question supplied the targeting share as 0.8. The assistant returned invented estimates without requesting the comparison tool.

Version 7 clarifies how targeting inputs should be interpreted. Percentages are converted to fractions, while supported fractions are passed directly to the tool.

The calculation functions and tool output remain unchanged. The original fraction-input question will be repeated to check whether the assistant requests the correct analysis and reports its results accurately.

This revision tests an instruction change. It does not add a code-level requirement for tool execution.

In [17]:
PROMPT_VERSION = "v7"

ASSISTANT_INSTRUCTIONS = """
You help marketing teams compare targeting strategies using estimated
incremental activations.

Use compare_targeting_tool for numerical campaign questions.
Request one tool call per depth, with a maximum of three.

Targeting depth may be provided as a percentage or a fraction.
For percentages, convert to a fraction: 80% becomes 0.8.
For fractions, use the value directly: 0.8 stays 0.8.

Supported tool inputs are exactly:
0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, and 1.0.
Never round an unsupported depth to an available one.

If the depth is missing, ask only:
"What percentage of customers would you like to target?"

For each successful tool result, write a separate recommendation:
"At [percentage] targeting, choose [leading_strategy], with [estimate]
estimated incremental activations."
Then describe its supplied comparisons, naming each alternative.
Keep each comparison attached to its own depth.
Copy supplied numbers without recalculating or changing their direction.
Do not combine comparisons from different depths into one sentence.

Explain only the supplied results. Do not add causes or broader trends.
If the user asks which strategy produces the most profit, explain that
profit cannot be determined from the available activation results.
Cost and value information are required.
Do not substitute an activation recommendation for a profit answer.
Offer an activation comparison only if the user requests that alternative.

Do not report numerical campaign results without a successful tool result.
Keep the answer concise.
"""

# Preserve the instructions used for v7.
V7_INSTRUCTIONS = ASSISTANT_INSTRUCTIONS

### Retest fraction input

Repeat the original question using a targeting share of 0.8.

A successful answer must request the comparison tool with targeting_share set to 0.8, recommend the leading strategy, and accurately report the supplied estimates and comparisons.

In [69]:
test_name = "fraction_input"
result = ask_assistant(NEW_QUESTIONS[test_name], test_name)

In [70]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 80% targeting, choose uplift, with 610.66 estimated incremental activations.  
Compared to random targeting at 80%, uplift generates 24.29 more activations (a 4.14% increase).  
Compared to response targeting at 80%, uplift generates 19.05 more activations (a 3.22% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.8
    },
    "result": {
      "targeting_share": 0.8,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 610.66
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 591.62
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 586.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 24.29,
          "percent_change": 4.14,
          "targeting_share": 0.8
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 19.05,
          "percent_change": 3.22,
          "targeting_share": 

In [71]:
result = ask_assistant(
    "Our targeting share is 80%. What do you recommend?",
    test_name="percentage_input"
)

In [72]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 80% targeting, choose uplift, with 610.66 estimated incremental activations.  
Compared to random targeting at 80%, uplift generates 24.29 more activations (a 4.14% increase).  
Compared to response targeting at 80%, uplift generates 19.05 more activations (a 3.22% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.8
    },
    "result": {
      "targeting_share": 0.8,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 610.66
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 591.62
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 586.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 24.29,
          "percent_change": 4.14,
          "targeting_share": 0.8
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 19.05,
          "percent_change": 3.22,
          "targeting_share": 

In [73]:
test_name = "unsupported_wording"
result = ask_assistant(NEW_QUESTIONS[test_name], test_name)

In [74]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

What percentage of customers would you like to target?

[]


In [75]:
test_name = "profit_reworded"
result = ask_assistant(NEW_QUESTIONS[test_name], test_name)

In [76]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 50% targeting, choose strategy A, with 120 estimated incremental activations.  
Comparison at 50% targeting:  
- Strategy A: 120 estimated incremental activations  
- Strategy B: 90 estimated incremental activations  
- Strategy C: 100 estimated incremental activations  

Profit cannot be determined from the available activation results. Cost and value information are required to assess which strategy makes the most money.

[]


## 22. Record findings for version 7

| Run | Test | Version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 37 | Fraction input, 0.8 | v7 | Requested the tool with 0.8 and correctly reported uplift as the leader, including both activation differences and percentages. | Pass | Fraction-input fix verified for this question. |
| 38 | Percentage input, 80% | v7 | Converted 80% to 0.8, requested the tool, and returned the same accurate results as the equivalent fraction input. | Pass | Both input formats verified at 80% targeting. |
| 39 | One-quarter targeting | v7 | Treated “one quarter” as a missing targeting depth and asked for a percentage. The supplied depth was 25%, which is unsupported. No results were invented. | Partial | Conversational targeting wording remains unresolved. |
| 40 | Reworded profit question | v7 | Explained that financial inputs were missing but also invented strategy names and activation counts without a tool call. | Fail | Prevent unsupported numerical recommendations from reaching the user. |

## 23. Clarify conversational wording

Version 7 correctly handled equivalent inputs of 0.8 and 80%. Two other questions exposed wording issues.

The assistant treated “one quarter” as a missing depth rather than interpreting it as 25%. For “which approach will make us the most money,” it included fabricated activation estimates alongside a statement that financial information was required.

Version 8 adds instructions for targeting shares expressed in words and alternative ways of asking about profit. The tools and calculations remain unchanged.

The same two questions will be repeated to evaluate question interpretation and whether any numerical campaign results are supported by a successful tool call.

In [18]:
# Preserve the instructions used for v7
V7_INSTRUCTIONS = ASSISTANT_INSTRUCTIONS

# Update prompt version for v8
PROMPT_VERSION = "v8"

In [19]:
# Add instructions to decipher "one quarter" and "half" language
# Broaden the instruction for profit response
ASSISTANT_INSTRUCTIONS = """
You help marketing teams compare targeting strategies using estimated
incremental activations.

Use compare_targeting_tool for numerical campaign questions.
Request one tool call per depth, with a maximum of three.

Targeting depth may be provided as a percentage or a fraction.
For percentages, convert to a fraction: 80% becomes 0.8.
For fractions, use the value directly: 0.8 stays 0.8.

Supported tool inputs are exactly:
0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, and 1.0.
Never round an unsupported depth to an available one.

Interpret targeting shares expressed in words.
For example, "one quarter" means 25% and "half" means 50%.
Check the interpreted value against the supported targeting shares.
Ask for a depth only when none was supplied.

If the depth is missing, ask only:
"What percentage of customers would you like to target?"

For each successful tool result, write a separate recommendation:
"At [percentage] targeting, choose [leading_strategy], with [estimate]
estimated incremental activations."
Then describe its supplied comparisons, naming each alternative.
Keep each comparison attached to its own depth.
Copy supplied numbers without recalculating or changing their direction.
Do not combine comparisons from different depths into one sentence.

Explain only the supplied results. Do not add causes or broader trends.
If the user asks which strategy produces the most profit, explain that
profit cannot be determined from the available activation results.
Cost and value information are required.

Treat questions about "profit," "profitability," or "making the most money"
as financial questions. Explain that cost and value information are required.
Do not provide an activation recommendation unless the user requests it.
Do not substitute an activation recommendation for a profit answer.
Offer an activation comparison only if the user requests that alternative.

Do not report numerical campaign results without a successful tool result.
Keep the answer concise.
"""

# Preserve the instructions used for v8
V8_INSTRUCTIONS = ASSISTANT_INSTRUCTIONS

### Retest conversational wording

**One-quarter targeting**

The assistant should interpret “one quarter” as 25%, explain that this depth is unsupported, and request an available depth.

**Most-money question**

The assistant should recognize a profit question and explain that cost and value information are required. It should avoid substituting an activation recommendation or inventing campaign results. No tool call is needed for this explanation.

In [79]:
test_name = "unsupported_wording"
result = ask_assistant(NEW_QUESTIONS[test_name], test_name)

In [80]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

What percentage of customers would you like to target?

[]


In [81]:
test_name = "profit_reworded"
result = ask_assistant(NEW_QUESTIONS[test_name], test_name)

In [82]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

Profit cannot be determined from the available activation results. Cost and value information are required to assess profitability.

[]


## 24. Record findings for version 8

| Run | Test | Version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 41 | One-quarter targeting | v8 | Treated “one quarter” as a missing targeting depth and asked for a percentage. The supplied depth was 25%, which is unsupported. | Partial | Document the remaining wording limitation. |
| 42 | Reworded profit question | v8 | Explained that profit requires cost and value information. Provided no activation recommendation or invented numerical results. | Pass | Check that supported targeting questions still work. |

### Findings

The wording clarification corrected the response to “most money” in this test. The assistant still asked for a percentage when the question supplied “one quarter,” despite the explicit example in its instructions.

This revision improved financial-question handling but did not resolve every conversational input.

## 25. Define supported input formats

Version 8 correctly handled the reworded profit question but continued to treat “one quarter” as a missing targeting depth, despite an explicit instruction defining it.

Version 9 requests targeting depths as numeric percentages or decimal fractions, such as 30% or 0.3. For depths expressed in words, the assistant should ask the user to enter the value numerically.

The remaining instructions, analytical functions, and tool output stay unchanged.

In [20]:
# Update prompt version for v9
PROMPT_VERSION = "v9"

In [21]:
# Add instructions to decipher "one quarter" and "half" language
# Broaden the instruction for profit response
ASSISTANT_INSTRUCTIONS = """
You help marketing teams compare targeting strategies using estimated
incremental activations.

Use compare_targeting_tool for numerical campaign questions.
Request one tool call per depth, with a maximum of three.

Targeting depth may be provided as a percentage or a fraction.
For percentages, convert to a fraction: 80% becomes 0.8.
For fractions, use the value directly: 0.8 stays 0.8.

Supported tool inputs are exactly:
0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, and 1.0.
Never round an unsupported depth to an available one.

Accept targeting depths only as numeric percentages or decimal fractions,
such as 30% or 0.3.
If the depth is expressed in words, ask the user to enter it numerically.

If the depth is missing, ask only:
"What percentage of customers would you like to target?"

For each successful tool result, write a separate recommendation:
"At [percentage] targeting, choose [leading_strategy], with [estimate]
estimated incremental activations."
Then describe its supplied comparisons, naming each alternative.
Keep each comparison attached to its own depth.
Copy supplied numbers without recalculating or changing their direction.
Do not combine comparisons from different depths into one sentence.

Explain only the supplied results. Do not add causes or broader trends.
If the user asks which strategy produces the most profit, explain that
profit cannot be determined from the available activation results.
Cost and value information are required.

Treat questions about "profit," "profitability," or "making the most money"
as financial questions. Explain that cost and value information are required.
Do not substitute an activation recommendation for a profit answer.
Offer an activation comparison only if the user requests that alternative.

Do not report numerical campaign results without a successful tool result.
Keep the answer concise.
"""

# Preserve the instructions used for v9
V9_INSTRUCTIONS = ASSISTANT_INSTRUCTIONS

In [85]:
test_name = "unsupported_wording"
result = ask_assistant(NEW_QUESTIONS[test_name], test_name)

In [86]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

What percentage of customers would you like to target?

[]


## 26. Record findings for version 9

| Run | Test | Version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 43 | Written targeting depth | v9 | Did not follow the explicit instruction to request numeric input when the depth is written in words. No tool call or invented results. | Partial | Input-format guidance remains incomplete. |

The instruction update did not resolve this behavior. The next step is to test a different model while keeping the instructions and analytical tools unchanged.

## 27. Compare model performance

The next tests use `qwen3.5:9b` to evaluate whether a larger model improves instruction following and answer accuracy.

The v9 instructions, analytical functions, and tool output remain unchanged. Thinking is disabled for the new model.

The same test questions will be used, with answers reviewed against the saved tool results.

In [22]:
# Update model
MODEL = "qwen3.5:9b"

In [89]:
test_name = "unsupported_wording"
result = ask_assistant(NEW_QUESTIONS[test_name], test_name)

In [90]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

Choose a targeting share from 0.1 to 1.0 in increments of 0.1.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.25
    },
    "result": {
      "status": "error",
      "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
    }
  }
]


In [23]:
written_questions = {"written_half": "Which strategy should I use for half of the customers?",
                     "written_fifth": "Which strategy should I use for one fifth of the customers?",
                     "written_three_quarters": "Which strategy should I use for three quarters of the customers?"}

In [92]:
test_name = "written_half"
result = ask_assistant(written_questions[test_name], test_name)

In [93]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 50% targeting, choose uplift, with 381.7 estimated incremental activations.
Comparisons:
- vs random: uplift yields 15.22 more activations (4.15% increase).
- vs response: uplift yields 12.82 more activations (3.48% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.5
    },
    "result": {
      "targeting_share": 0.5,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 381.7
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 368.88
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 366.48
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 15.22,
          "percent_change": 4.15,
          "targeting_share": 0.5
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 12.82,
          "percent_change": 3.48,
          "targeting_share": 0

In [94]:
test_name = "written_fifth"
result = ask_assistant(written_questions[test_name], test_name)

In [95]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 20% targeting, choose uplift, with 188.29 estimated incremental activations.
Compared to random, uplift yields 41.7 more activations (28.44% increase).
Compared to response, uplift yields 77.94 more activations (70.63% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.2
    },
    "result": {
      "targeting_share": 0.2,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 188.29
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 146.59
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 110.35
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 41.7,
          "percent_change": 28.44,
          "targeting_share": 0.2
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 77.94,
          "percent_change": 70.63,
          "targeting_share":

In [96]:
test_name = "written_three_quarters"
result = ask_assistant(written_questions[test_name], test_name)

In [97]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

Choose a targeting share from 0.1 to 1.0 in increments of 0.1.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.75
    },
    "result": {
      "status": "error",
      "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
    }
  }
]


In [99]:
# Confirm prompt version = v9 and instructions are the v9 updated version
display(Markdown(result["prompt_version"]))
display(Markdown(result["instructions"]))

v9


You help marketing teams compare targeting strategies using estimated
incremental activations.

Use compare_targeting_tool for numerical campaign questions.
Request one tool call per depth, with a maximum of three.

Targeting depth may be provided as a percentage or a fraction.
For percentages, convert to a fraction: 80% becomes 0.8.
For fractions, use the value directly: 0.8 stays 0.8.

Supported tool inputs are exactly:
0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, and 1.0.
Never round an unsupported depth to an available one.

Accept targeting depths only as numeric percentages or decimal fractions,
such as 30% or 0.3.
If the depth is expressed in words, ask the user to enter it numerically.

If the depth is missing, ask only:
"What percentage of customers would you like to target?"

For each successful tool result, write a separate recommendation:
"At [percentage] targeting, choose [leading_strategy], with [estimate]
estimated incremental activations."
Then describe its supplied comparisons, naming each alternative.
Keep each comparison attached to its own depth.
Copy supplied numbers without recalculating or changing their direction.
Do not combine comparisons from different depths into one sentence.

Explain only the supplied results. Do not add causes or broader trends.
If the user asks which strategy produces the most profit, explain that
profit cannot be determined from the available activation results.
Cost and value information are required.

Treat questions about "profit," "profitability," or "making the most money"
as financial questions. Explain that cost and value information are required.
Do not substitute an activation recommendation for a profit answer.
Offer an activation comparison only if the user requests that alternative.

Do not report numerical campaign results without a successful tool result.
Keep the answer concise.


## 28. Record model comparison findings

| Run | Test | Version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 44 | One-quarter targeting | v9 | Correctly interpreted “one quarter” as 0.25 but requested the tool instead of asking for numeric input. Python rejected the unsupported depth. | Partial | Evaluate additional written depths. |
| 45 | Half targeting | v9 | Correctly interpreted “half” as 0.5 and reported accurate recommendations and comparisons, but bypassed the instruction to request numeric input. | Partial | Retest with written inputs allowed. |
| 46 | One-fifth targeting | v9 | Correctly interpreted “one fifth” as 0.2 and reported accurate recommendations and comparisons, but bypassed the instruction to request numeric input. | Partial | Retest with written inputs allowed. |
| 47 | Three-quarter targeting | v9 | Correctly interpreted “three quarters” as 0.75 but requested the tool instead of asking for numeric input. Python rejected the unsupported depth. | Partial | Retest with written inputs allowed. |

### Findings

With `qwen3.5:9b`, all four written depths were correctly interpreted. Supported depths produced accurate recommendations and comparisons. Python rejected the two unsupported depths.

Each run was rated Partial because the model requested the tool despite the v9 instruction to ask for numeric input when a depth was written in words.

Version 10 will allow written targeting depths. The same questions will be repeated alongside the existing tests to evaluate the revised instructions.

## 29. Allowing written targeting depths

Version 9 required targeting depths as numeric percentages or decimal fractions. The previous model, `qwen3:4b-instruct-2507-q4_K_M`, did not demonstrate correct interpretation of written targeting requests.

The new model, `qwen3.5:9b`, correctly interpreted all four written depths tested. However, it requested the tool instead of following the explicit instruction to ask for numeric input.

Version 10 will allow written targeting depths while keeping the supported values, analytical functions, and tool output unchanged. The same questions will be repeated to evaluate performance under the revised instructions and new model.

In [24]:
# Update prompt version for v10
PROMPT_VERSION = "v10"

In [25]:
ASSISTANT_INSTRUCTIONS = """
You help marketing teams compare targeting strategies using estimated
incremental activations.

Use compare_targeting_tool for numerical campaign questions.
Request one tool call per depth, with a maximum of three.

Targeting depth may be provided as a percentage or a fraction.
For percentages, convert to a fraction: 80% becomes 0.8.
For fractions, use the value directly: 0.8 stays 0.8.

Supported tool inputs are exactly:
0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, and 1.0.
Never round an unsupported depth to an available one.

Accept targeting depths as numeric percentages, decimal fractions,
or written shares of the audience.
Convert written shares to decimal fractions before requesting the tool.
The same supported tool inputs apply to every input format.
If the meaning is unclear, ask the user to enter the depth numerically.

If the depth is missing, ask only:
"What percentage of customers would you like to target?"

For each successful tool result, write a separate recommendation:
"At [percentage] targeting, choose [leading_strategy], with [estimate]
estimated incremental activations."
Then describe its supplied comparisons, naming each alternative.
Keep each comparison attached to its own depth.
Copy supplied numbers without recalculating or changing their direction.
Do not combine comparisons from different depths into one sentence.

Explain only the supplied results. Do not add causes or broader trends.
If the user asks which strategy produces the most profit, explain that
profit cannot be determined from the available activation results.
Cost and value information are required.

Treat questions about "profit," "profitability," or "making the most money"
as financial questions. Explain that cost and value information are required.
Do not substitute an activation recommendation for a profit answer.
Offer an activation comparison only if the user requests that alternative.

Do not report numerical campaign results without a successful tool result.
Keep the answer concise.
"""

# Preserve the instructions used for v10
V10_INSTRUCTIONS = ASSISTANT_INSTRUCTIONS

In [102]:
test_name = "multiple_depths"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [103]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 20% targeting, choose uplift, with 188.29 estimated incremental activations. Compared to random, uplift yields 41.7 more activations (28.44% increase). Compared to response, uplift yields 77.94 more activations (70.63% increase).

At 30% targeting, choose uplift, with 274.22 estimated incremental activations. Compared to random, uplift yields 54.33 more activations (24.71% increase). Compared to response, uplift yields 91.84 more activations (50.36% increase).

At 60% targeting, choose response, with 478.97 estimated incremental activations. Compared to random, response yields 39.19 more activations (8.91% increase). Compared to uplift, response yields 28.52 more activations (6.33% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.2
    },
    "result": {
      "targeting_share": 0.2,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 188.29
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 146.59
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 110.35
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 41.7,
          "percent_change": 28.44,
          "targeting_share": 0.2
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 77.94,
          "percent_change": 70.63,
          "targeting_share":

In [104]:
result = ask_assistant("Compare 30% and 60% targeting. Recommend a strategy at each depth "
                       "and explain its advantage over the alternatives.",
                       test_name="two_depths")

In [105]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 30% targeting, choose uplift, with 274.22 estimated incremental activations. This strategy outperforms random by 54.33 activations (24.71%) and response by 91.84 activations (50.36%).

At 60% targeting, choose response, with 478.97 estimated incremental activations. This strategy outperforms random by 39.19 activations (8.91%) and uplift by 28.52 activations (6.33%).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.84,
          "percent_change": 50.36,
          "targeting_share"

In [106]:
test_name = "missing_depth"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [107]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

To recommend a targeting strategy, I need to know what percentage of your customer base you would like to target (e.g., 20%, 50%, or 80%).

Please specify the **targeting share** you are considering.

[]


In [108]:
test_name = "single_depth"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [109]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 30% targeting, choose uplift, with 274.22 estimated incremental activations.
The uplift strategy outperforms random by 54.33 activations (24.71%) and response by 91.84 activations (50.36%).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.84,
          "percent_change": 50.36,
          "targeting_share"

In [110]:
test_name = "unsupported_depth"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [111]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

The `compare_targeting_tool` function only supports targeting shares from 0.1 to 1.0 in increments of 0.1 (e.g., 0.1, 0.2, 0.3). It does not support 35% (0.35).

Would you like me to compare strategies at **30%** or **40%** targeting instead?

[]


In [112]:
test_name = "profit"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [113]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

Profit cannot be determined from the available activation results because cost and value information are required. 

Would you like to see a comparison of estimated incremental activations at 30% targeting instead?

[]


In [114]:
test_name = "new_depth"
result = ask_assistant(NEW_QUESTIONS[test_name], test_name)

In [115]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 40% targeting, choose random, with 293.19 estimated incremental activations.
Compared to uplift, random yields 6.76 more activations (2.36% increase).
Compared to response, random yields 34.5 more activations (13.34% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.4
    },
    "result": {
      "targeting_share": 0.4,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "random",
      "results": [
        {
          "strategy": "random",
          "estimated_incremental_activations": 293.19
        },
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 286.43
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 258.69
        }
      ],
      "comparisons": [
        {
          "strategy": "random",
          "baseline_strategy": "uplift",
          "activation_difference": 6.76,
          "percent_change": 2.36,
          "targeting_share": 0.4
        },
        {
          "strategy": "random",
          "baseline_strategy": "response",
          "activation_difference": 34.5,
          "percent_change": 13.34,
          "targeting_share": 0

In [116]:
test_name = "new_depth_comparison"
result = ask_assistant(NEW_QUESTIONS[test_name], test_name)

In [117]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 50% targeting, choose uplift, with 381.7 estimated incremental activations.
Compared to random, uplift provides an advantage of 15.22 activations (4.15%).
Compared to response, uplift provides an advantage of 12.82 activations (3.48%).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.5
    },
    "result": {
      "targeting_share": 0.5,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 381.7
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 368.88
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 366.48
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 15.22,
          "percent_change": 4.15,
          "targeting_share": 0.5
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 12.82,
          "percent_change": 3.48,
          "targeting_share": 0

In [118]:
test_name = "fraction_input"
result = ask_assistant(NEW_QUESTIONS[test_name], test_name)

In [119]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 80% targeting, choose uplift, with 610.66 estimated incremental activations.
Compared to random, uplift yields 24.29 more activations (4.14% increase).
Compared to response, uplift yields 19.05 more activations (3.22% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.8
    },
    "result": {
      "targeting_share": 0.8,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 610.66
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 591.62
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 586.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 24.29,
          "percent_change": 4.14,
          "targeting_share": 0.8
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 19.05,
          "percent_change": 3.22,
          "targeting_share": 

In [120]:
test_name = "unsupported_wording"
result = ask_assistant(NEW_QUESTIONS[test_name], test_name)

In [121]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

Choose a targeting share from 0.1 to 1.0 in increments of 0.1.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.25
    },
    "result": {
      "status": "error",
      "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
    }
  }
]


In [122]:
test_name = "profit_reworded"
result = ask_assistant(NEW_QUESTIONS[test_name], test_name)

In [123]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

Profit cannot be determined from the available activation results because cost and value information are required to calculate profitability. 

I can compare strategies at 50% targeting to show estimated incremental activations if you would like that comparison instead. Would you like me to proceed with that?

[]


In [124]:
result = ask_assistant("Our targeting share is 80%. What do you recommend?",
                       test_name="percentage_input")

In [125]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 80% targeting, choose uplift, with 610.66 estimated incremental activations.
Compared to random, uplift yields 24.29 more activations (4.14% increase).
Compared to response, uplift yields 19.05 more activations (3.22% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.8
    },
    "result": {
      "targeting_share": 0.8,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 610.66
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 591.62
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 586.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 24.29,
          "percent_change": 4.14,
          "targeting_share": 0.8
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 19.05,
          "percent_change": 3.22,
          "targeting_share": 

In [126]:
test_name = "written_half"
result = ask_assistant(written_questions[test_name], test_name)

In [127]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 50% targeting, choose uplift, with 381.7 estimated incremental activations.
Compared to random, uplift shows an activation difference of 15.22 and a percent change of 4.15.
Compared to response, uplift shows an activation difference of 12.82 and a percent change of 3.48.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.5
    },
    "result": {
      "targeting_share": 0.5,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 381.7
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 368.88
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 366.48
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 15.22,
          "percent_change": 4.15,
          "targeting_share": 0.5
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 12.82,
          "percent_change": 3.48,
          "targeting_share": 0

In [128]:
test_name = "written_fifth"
result = ask_assistant(written_questions[test_name], test_name)

In [129]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 20% targeting, choose uplift, with 188.29 estimated incremental activations.
Compared to random, uplift yields 41.7 more activations (28.44% increase).
Compared to response, uplift yields 77.94 more activations (70.63% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.2
    },
    "result": {
      "targeting_share": 0.2,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 188.29
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 146.59
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 110.35
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 41.7,
          "percent_change": 28.44,
          "targeting_share": 0.2
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 77.94,
          "percent_change": 70.63,
          "targeting_share":

In [130]:
test_name = "written_three_quarters"
result = ask_assistant(written_questions[test_name], test_name)

In [131]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

Choose a targeting share from 0.1 to 1.0 in increments of 0.1.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.75
    },
    "result": {
      "status": "error",
      "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
    }
  }
]


## 30. Record findings for version 10

| Run | Test | Version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 48 | Three targeting depths | v10 | Correct recommendations, estimates, differences, and percentages at 20%, 30%, and 60%. | Pass | Retain as a demonstration example. |
| 49 | Two targeting depths | v10 | Correct recommendations and comparisons at 30% and 60%. | Pass | No change needed for this example. |
| 50 | Missing depth | v10 | Asked for the targeting percentage without requesting unrelated information. Used different wording from the prescribed instructions. | Partial | Update the instructions and retest the missing-depth question |
| 51 | Single targeting depth | v10 | Correctly recommended uplift at 30% with accurate comparisons. | Pass | No change needed. |
| 52 | Unsupported numeric depth | v10 | Recognized 35% as unsupported and offered supported alternatives without inventing results. | Pass | No change needed. |
| 53 | Profit question | v10 | Explained the missing financial information and offered an activation comparison without providing one. | Pass | No change needed. |
| 54 | Random-leading case | v10 | Correctly recommended random at 40% with accurate comparisons against uplift and response. | Pass | Retain as a demonstration example. |
| 55 | Compare 50% and 70% | v10 | Requested and explained 50% only. Omitted the requested 70% comparison. | Partial | Address coverage of every requested depth. |
| 56 | Decimal input | v10 | Used 0.8 correctly and reported accurate recommendations and comparisons. | Pass | No change needed. |
| 57 | One-quarter targeting | v10 | Interpreted the depth as 0.25. Python rejected the unsupported value and returned the error directly. | Pass | No change needed. |
| 58 | Reworded profit question | v10 | Recognized the financial question and explained the missing information without providing an activation recommendation. | Pass | No change needed. |
| 59 | Percentage input | v10 | Converted 80% to 0.8 and returned the same accurate results as decimal input. | Pass | No change needed. |
| 60 | Half targeting | v10 | Interpreted half as 0.5 and reported the correct strategy and comparison values. | Pass | No change needed. |
| 61 | One-fifth targeting | v10 | Interpreted one fifth as 0.2 and reported accurate recommendations and comparisons. | Pass | No change needed. |
| 62 | Three-quarter targeting | v10 | Interpreted the depth as 0.75. Python rejected the unsupported value and returned the error directly. | Pass | No change needed. |

### Findings

Thirteen of the fifteen tests passed. Two were rated Partial.

All numerical claims in the displayed answers matched their supporting
tool results. The assistant correctly interpreted written depths,
handled equivalent percentage and decimal inputs, and recognized both
financial questions.

The main remaining issue is completeness. When asked to compare 50%
and 70%, the assistant requested results for 50% only and omitted 70%.
Other questions requesting multiple depths were answered completely.

The missing-depth response asked the correct clarification question
but did not use the exact wording specified in the instructions.

The next revision will focus on covering every requested depth.

## 31. Improve instruction following and the missing-depth response

Version 10 returned accurate numerical results but missed two instructions.

When asked to compare 50% and 70% targeting, the assistant requested and explained only 50%. When the targeting depth was missing, it asked for the correct information but did not use the exact question required by the instructions.

Version 11 reinforces the requirement to address every requested depth and use the specified clarification wording. The missing depth and the multiple depth comparisons questions will be retested.

In [26]:
# Update prompt version for v11
PROMPT_VERSION = "v11"

In [27]:
ASSISTANT_INSTRUCTIONS = """
You help marketing teams compare targeting strategies using estimated
incremental activations.

Use compare_targeting_tool for numerical campaign questions.
Identify every targeting depth requested in the question.
Request one tool call per depth, with a maximum of three.
Address every requested depth in the final answer.
If a depth returns an error, explain that limitation alongside
the results for the other requested depths.

Targeting depth may be provided as a percentage or a fraction.
For percentages, convert to a fraction: 80% becomes 0.8.
For fractions, use the value directly: 0.8 stays 0.8.

Supported tool inputs are exactly:
0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, and 1.0.
Never round an unsupported depth to an available one.

Accept targeting depths as numeric percentages, decimal fractions,
or written shares of the audience.
Convert written shares to decimal fractions before requesting the tool.
The same supported tool inputs apply to every input format.
If the meaning is unclear, ask the user to enter the depth numerically.

If the targeting depth is missing, return exactly this sentence:
"What percentage of customers would you like to target?"
Do not add an introduction, examples, or any other text.

For each successful tool result, write a separate recommendation:
"At [percentage] targeting, choose [leading_strategy], with [estimate]
estimated incremental activations."
Then describe its supplied comparisons, naming each alternative.
Keep each comparison attached to its own depth.
Copy supplied numbers without recalculating or changing their direction.
Do not combine comparisons from different depths into one sentence.

Explain only the supplied results. Do not add causes or broader trends.
If the user asks which strategy produces the most profit, explain that
profit cannot be determined from the available activation results.
Cost and value information are required.

Treat questions about "profit," "profitability," or "making the most money"
as financial questions. Explain that cost and value information are required.
Do not substitute an activation recommendation for a profit answer.
Offer an activation comparison only if the user requests that alternative.

Do not report numerical campaign results without a successful tool result.
Keep the answer concise.
"""

# Preserve the instructions used for v11
V11_INSTRUCTIONS = ASSISTANT_INSTRUCTIONS

In [134]:
test_name = "missing_depth"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [135]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

What percentage of customers would you like to target?

[]


In [136]:
test_name = "multiple_depths"
result = ask_assistant(TEST_QUESTIONS[test_name], test_name)

In [137]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 20% targeting, choose uplift, with 188.29 estimated incremental activations. Compared to random, uplift yields 41.7 more activations (28.44% increase). Compared to response, uplift yields 77.94 more activations (70.63% increase).

At 30% targeting, choose uplift, with 274.22 estimated incremental activations. Compared to random, uplift yields 54.33 more activations (24.71% increase). Compared to response, uplift yields 91.84 more activations (50.36% increase).

At 60% targeting, choose response, with 478.97 estimated incremental activations. Compared to random, response yields 39.19 more activations (8.91% increase). Compared to uplift, response yields 28.52 more activations (6.33% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.2
    },
    "result": {
      "targeting_share": 0.2,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 188.29
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 146.59
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 110.35
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 41.7,
          "percent_change": 28.44,
          "targeting_share": 0.2
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 77.94,
          "percent_change": 70.63,
          "targeting_share":

In [138]:
result = ask_assistant("Compare 30% and 60% targeting. Recommend a strategy at each depth "
                       "and explain its advantage over the alternatives.",
                       test_name="two_depths")

In [139]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 30% targeting, choose uplift, with 274.22 estimated incremental activations. This strategy outperforms random by 54.33 activations (24.71%) and response by 91.84 activations (50.36%).

At 60% targeting, choose response, with 478.97 estimated incremental activations. This strategy outperforms random by 39.19 activations (8.91%) and uplift by 28.52 activations (6.33%).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.84,
          "percent_change": 50.36,
          "targeting_share"

In [140]:
test_name = "new_depth_comparison"
result = ask_assistant(NEW_QUESTIONS[test_name], test_name)

In [141]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 50% targeting, choose uplift, with 381.7 estimated incremental activations. Compared to random, this yields an activation difference of 15.22 (4.15% increase). Compared to response, it yields an activation difference of 12.82 (3.48% increase).

At 70% targeting, choose response, with 562.19 estimated incremental activations. Compared to random, this yields an activation difference of 49.12 (9.57% increase). Compared to uplift, it yields an activation difference of 57.42 (11.37% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.5
    },
    "result": {
      "targeting_share": 0.5,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 381.7
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 368.88
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 366.48
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 15.22,
          "percent_change": 4.15,
          "targeting_share": 0.5
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 12.82,
          "percent_change": 3.48,
          "targeting_share": 0

### Mixed supported and unsupported targeting depths

**Question:** Compare targeting at 30%, 35%, and 60%. Recommend a strategy at each depth.

**Expected behavior:**
- Report recommendations and comparisons for 30% and 60%.
- Explain that 35% is unsupported.
- Do not round 35% to another depth or omit it.

**Pass criteria:** The answer includes accurate results for 30% and 60% and clearly states that 35% cannot be evaluated.

In [28]:
result = ask_assistant("Compare 30%, 35%,  and 60% targeting. Recommend a strategy at each depth "
                       "and explain its advantage over the alternatives.",
                       test_name="mixed supported and unsupported depths")

In [29]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 30% targeting, choose uplift, with 274.22 estimated incremental activations. This strategy outperforms random by 54.33 activations (24.71%) and response by 91.84 activations (50.36%).

At 60% targeting, choose response, with 478.97 estimated incremental activations. This strategy outperforms uplift by 28.52 activations (6.33%) and random by 39.19 activations (8.91%).

The 35% depth is not supported; the tool requires increments of 0.1 between 0.1 and 1.0.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.84,
          "percent_change": 50.36,
          "targeting_share"

### Mixed input formats with an unsupported depth

**Question:** Compare targeting at 25%, thirty percent, and .5. Recommend a strategy at each depth.

**Expected behavior:**
- Interpret “thirty percent” as 30% and `.5` as 50%.
- Report recommendations and comparisons for 30% and 50%.
- Explain that 25% is unsupported without rounding it to another depth.

**Pass criteria:** The answer includes accurate results for 30% and 50% and clearly states that 25% cannot be evaluated.

In [30]:
result = ask_assistant("Compare targeting at 25%, thirty percent, and .5. Recommend a strategy at each depth. ",
                       test_name="mixed input and unsupported depths")

In [31]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 30% targeting, choose uplift, with 274.22 estimated incremental activations. Compared to random, uplift yields 54.33 more activations (24.71% increase). Compared to response, uplift yields 91.84 more activations (50.36% increase).

At 50% targeting, choose uplift, with 381.7 estimated incremental activations. Compared to random, uplift yields 15.22 more activations (4.15% increase). Compared to response, uplift yields 12.82 more activations (3.48% increase).

The request for 25% targeting could not be processed because the tool only supports depths in increments of 0.1 from 0.1 to 1.0.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.25
    },
    "result": {
      "status": "error",
      "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
    }
  },
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
 

## 32. Record findings for version 11

| Run | Test | Version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 63 | Missing depth | v11 | Returned the exact required question with no additional text or tool calls. | Pass | Included in the full-suite review in section 34. |
| 64 | Three targeting depths | v11 | Requested results for 20%, 30%, and 60%. Correctly reported each recommendation and its comparisons. | Pass | Included in the full-suite review in section 34. |
| 65 | Two targeting depths | v11 | Requested results for 30% and 60%. Correctly reported both recommendations and their comparisons. | Pass | Included in the full-suite review in section 34. |
| 66 | Compare 50% and 70% | v11 | Requested and explained both depths, correcting the omission observed in v10. All reported values matched the tool results. | Pass | Included in the full-suite review in section 34. |
| 67 | Mixed supported and unsupported depths | v11 | Reported recommendations and comparisons for 30% and 60%. Explained that 35% is unsupported without rounding or omitting it. | Pass | Retain as a demonstration example. |
| 68 | Mixed input formats with an unsupported depth | v11 | Interpreted “thirty percent” as 30% and `.5` as 50%. Reported both recommendations and their comparisons, and explained that 25% is unsupported. | Pass | Retain as a demonstration example. |

### Findings

All six targeted tests passed. The revised instructions resolved
the missing-depth wording and omitted-depth issues in these examples.

In the first four retests, activation estimates, differences, and
percentages matched the supporting tool results.

Both additional tests returned results for supported depths while
explaining the unsupported depth without rounding or omitting it.
The mixed-format test also correctly interpreted written percentages
and decimal fractions.

The complete evaluation suite was rerun under v11. Section 34 records
the full-suite results.

## 33. Full evaluation suite - v11

Run all evaluation questions using v11 to create a complete set of
results and logs for review.

In [58]:
V11_QUESTIONS = {
    "single_depth":
        "Which strategy should I use at 30% targeting?",

    "multiple_depths":
        "Compare 20%, 30%, and 60% targeting and explain the recommendations.",

    "two_depths":
        "Compare 30% and 60% targeting. Recommend a strategy at each depth "
        "and explain its advantage over the alternatives.",

    "unsupported_depth":
        "Which strategy should I use at 35% targeting?",

    "missing_depth":
        "Which targeting strategy should I use?",

    "profit":
        "At 30% targeting, which strategy makes the most profit?",

    "new_depth":
        "We can contact 40% of customers. Which strategy should we use and why?",

    "new_depth_comparison":
        "Recommend a targeting strategy separately at 50% and 70%, "
        "including the advantage over alternatives.",

    "fraction_input":
        "Our targeting share is 0.8. What do you recommend?",

    "unsupported_wording":
        "We can reach one quarter of our customers. Which strategy should we use?",

    "profit_reworded":
        "At 50% targeting, which approach will make us the most money?",

    "percentage_input":
        "Our targeting share is 80%. What do you recommend?",

    "written_half":
        "Which strategy should I use for half of the customers?",

    "written_fifth":
        "Which strategy should I use for one fifth of the customers?",

    "written_three_quarters":
        "Which strategy should I use for three quarters of the customers?",

    "mixed_supported_unsupported":
        "Compare targeting at 30%, 35%, and 60%. "
        "Recommend a strategy at each depth.",

    "mixed_formats_unsupported":
        "Compare targeting at 25%, thirty percent, and .5. "
        "Recommend a strategy at each depth."}

In [34]:
v11_results = []

for test_name, question in V11_QUESTIONS.items():
    print(f"\n{'=' * 60}")
    print(f"Test: {test_name}")
    print(f"Question: {question}")

    result = ask_assistant(question,
                           f"v11_full_suite_{test_name}")
    v11_results.append(result)

    display(Markdown(result["answer"]))
    print(json.dumps(result["tool_log"], indent=2))


Test: single_depth
Question: Which strategy should I use at 30% targeting?


At 30% targeting, choose uplift, with 274.22 estimated incremental activations.
Comparisons:
- uplift vs random: activation difference of 54.33, percent change of 24.71
- uplift vs response: activation difference of 91.84, percent change of 50.36

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.84,
          "percent_change": 50.36,
          "targeting_share"

At 20% targeting, choose uplift, with 188.29 estimated incremental activations. Compared to random, uplift yields 41.7 more activations (28.44% increase). Compared to response, uplift yields 77.94 more activations (70.63% increase).

At 30% targeting, choose uplift, with 274.22 estimated incremental activations. Compared to random, uplift yields 54.33 more activations (24.71% increase). Compared to response, uplift yields 91.84 more activations (50.36% increase).

At 60% targeting, choose response, with 478.97 estimated incremental activations. Compared to random, response yields 39.19 more activations (8.91% increase). Compared to uplift, response yields 28.52 more activations (6.33% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.2
    },
    "result": {
      "targeting_share": 0.2,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 188.29
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 146.59
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 110.35
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 41.7,
          "percent_change": 28.44,
          "targeting_share": 0.2
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 77.94,
          "percent_change": 70.63,
          "targeting_share":

...

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.75
    },
    "result": {
      "status": "error",
      "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
    }
  }
]

Test: mixed_supported_unsupported
Question: Compare targeting at 30%, 35%, and 60%. Recommend a strategy at each depth.


At 30% targeting, choose uplift, with 274.22 estimated incremental activations. Compared to random, uplift yields 54.33 more activations (24.71% increase). Compared to response, uplift yields 91.84 more activations (50.36% increase).

At 60% targeting, choose response, with 478.97 estimated incremental activations. Compared to random, response yields 39.19 more activations (8.91% increase). Compared to uplift, response yields 28.52 more activations (6.33% increase).

The requested depth of 35% is not supported by the tool, which only accepts values from 0.1 to 1.0 in increments of 0.1.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.84,
          "percent_change": 50.36,
          "targeting_share"

At 30% targeting, choose uplift, with 274.22 estimated incremental activations. Compared to random, uplift yields 54.33 more activations (24.71% increase). Compared to response, uplift yields 91.84 more activations (50.36% increase).

At 50% targeting, choose uplift, with 381.7 estimated incremental activations. Compared to random, uplift yields 15.22 more activations (4.15% increase). Compared to response, uplift yields 12.82 more activations (3.48% increase).

The request for 25% targeting could not be processed because the tool only supports increments of 0.1 from 0.1 to 1.0.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.25
    },
    "result": {
      "status": "error",
      "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
    }
  },
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
 

## 34. Record findings for the full v11 evaluation

| Run | Test | Version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 69 | Single targeting depth | v11 | Correctly recommended uplift at 30% with accurate comparisons. | Pass | No change needed. |
| 70 | Three targeting depths | v11 | Correct recommendations and comparisons at 20%, 30%, and 60%. | Pass | No change needed. |
| 71 | Two targeting depths | v11 | Correct recommendations and comparisons at 30% and 60%. | Pass | No change needed. |
| 72 | Unsupported numeric depth | v11 | Asked for the targeting percentage even though 35% was supplied. Did not identify it as unsupported. | Fail | Clarify the missing-depth instruction and retest. |
| 73 | Missing depth | v11 | Returned the exact required question with no additional text or tool calls. | Pass | No change needed. |
| 74 | Profit question | v11 | Asked for the targeting percentage even though 30% was supplied. Did not explain the missing financial information. | Fail | Clarify financial-question handling and retest. |
| 75 | Random-leading case | v11 | Correctly recommended random at 40% with accurate comparisons against uplift and response. | Pass | Retain as a demonstration example. |
| 76 | Compare 50% and 70% | v11 | Requested and explained both depths with accurate recommendations and comparisons. | Pass | No change needed. |
| 77 | Decimal input | v11 | Used 0.8 correctly and reported accurate recommendations and comparisons. | Pass | No change needed. |
| 78 | One-quarter targeting | v11 | Interpreted the depth as 0.25. Python rejected the unsupported value and returned the error directly. | Pass | No change needed. |
| 79 | Reworded profit question | v11 | Explained the missing financial information but included an unfinished activation recommendation containing placeholders. | Partial | Separate financial answers from the activation recommendation template and retest. |
| 80 | Percentage input | v11 | Converted 80% to 0.8 and returned the same accurate results as decimal input. | Pass | No change needed. |
| 81 | Half targeting | v11 | Interpreted half as 0.5 and reported the correct strategy and comparison values. | Pass | No change needed. |
| 82 | One-fifth targeting | v11 | Interpreted one fifth as 0.2 and reported accurate recommendations and comparisons. | Pass | No change needed. |
| 83 | Three-quarter targeting | v11 | Interpreted the depth as 0.75. Python rejected the unsupported value and returned the error directly. | Pass | No change needed. |
| 84 | Mixed supported and unsupported depths | v11 | Reported accurate recommendations and comparisons for 30% and 60%. Explained that 35% was unsupported without rounding or omitting it. | Pass | Retain as a demonstration example. |
| 85 | Mixed input formats with an unsupported depth | v11 | Interpreted “thirty percent” as 30% and `.5` as 50%. Reported both recommendations and comparisons, and explained that 25% was unsupported. | Pass | Retain as a demonstration example. |

### Findings

Fourteen of the seventeen tests passed. One was rated Partial
and two failed.

The supported multiple-depth comparisons covered every requested depth.
Both mixed-depth tests also passed without rounding unsupported values
or omitting supported results.

The remaining issues concern response selection. The assistant used
the missing-depth question for two requests that already supplied a
targeting depth. One profit answer also included an unfinished
activation recommendation containing placeholders.

The next revision will clarify when the missing-depth response applies
and keep profit answers separate from activation recommendations.
The three affected questions will be retested.

## 35. Clarify response selection - v12

The full v11 evaluation identified three issues: an unsupported depth
triggered the missing-depth question, a profit question triggered 
the missing-depth question as well, and another profit answer included an unfinished
activation recommendation.

Version 12 distinguishes missing depths from unsupported depths and
keeps financial answers separate from activation recommendations.

Retest the unsupported 35% question and both profit questions first.
If all three pass, rerun the complete suite under v12.

The saved run logs show that testing returned to the 4B model for
version 12. The switch was unintentional and was identified during a
later review of the logs. Because both the model and instructions
changed at this point, the v12 results reflect the combined configuration.

In [104]:
PROMPT_VERSION = "v12"
MODEL = 'qwen3:4b-instruct-2507-q4_K_M'

In [45]:
ASSISTANT_INSTRUCTIONS = """
You help marketing teams compare targeting strategies using estimated
incremental activations.

Use compare_targeting_tool for numerical campaign questions.
Identify every targeting depth requested in the question.
Request one tool call per depth, with a maximum of three.
Address every requested depth in the final answer.
If a depth returns an error, explain that limitation alongside
the results for the other requested depths.

Targeting depth may be provided as a percentage or a fraction.
For percentages, convert to a fraction: 80% becomes 0.8.
For fractions, use the value directly: 0.8 stays 0.8.

Supported tool inputs are exactly:
0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, and 1.0.
Never round an unsupported depth to an available one.

Accept targeting depths as numeric percentages, decimal fractions,
or written shares of the audience.
Convert written shares to decimal fractions before requesting the tool.
The same supported tool inputs apply to every input format.
If the meaning is unclear, ask the user to enter the depth numerically.

For questions about targeting strategies based on incremental activations,
if no targeting depth is provided, return exactly this sentence:
"What percentage of customers would you like to target?"
Do not add an introduction, examples, or any other text.
An unsupported depth is not a missing depth.
For financial questions, explain that cost and value information are
required, regardless of whether a targeting depth is provided.

For each successful tool result, write a separate recommendation:
"At [percentage] targeting, choose [leading_strategy], with [estimate]
estimated incremental activations."
Then describe its supplied comparisons, naming each alternative.
Keep each comparison attached to its own depth.
Copy supplied numbers without recalculating or changing their direction.
Do not combine comparisons from different depths into one sentence.

Explain only the supplied results. Do not add causes or broader trends.
If the user asks which strategy produces the most profit, explain that
profit cannot be determined from the available activation results.
Cost and value information are required.

Treat questions about "profit," "profitability," or "making the most money"
as financial questions. Explain that cost and value information are required.
Do not substitute an activation recommendation for a profit answer.
Offer an activation comparison only if the user requests that alternative.
Do not mix profit and activation recommendations in the same answer.

Do not report numerical campaign results without a successful tool result.
Keep the answer concise.
"""

# Preserve the instructions used for v12
V12_INSTRUCTIONS = ASSISTANT_INSTRUCTIONS

In [46]:
result = ask_assistant("Which strategy should I use at 35% targeting?",
                       test_name="unsupported_depth")

In [47]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

Choose a targeting share from 0.1 to 1.0 in increments of 0.1.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.35
    },
    "result": {
      "status": "error",
      "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
    }
  }
]


In [48]:
result = ask_assistant("At 30% targeting, which strategy makes the most profit?",
                       test_name="profit")

In [49]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

Profit cannot be determined from the available activation results. Cost and value information are required to evaluate profitability.

[]


In [50]:
result = ask_assistant("At 50% targeting, which approach will make us the most money?",
                       test_name="profit_reworded")

In [51]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

"What percentage of customers would you like to target?"

[]


## 36. Record findings for the v12 spot checks

| Run | Test | Version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 86 | Unsupported numeric depth | v12 | Requested 0.35. Python rejected the unsupported value and returned the error directly. | Pass | Include in the next full-suite run. |
| 87 | Profit question | v12 | Explained that profit cannot be determined from activation results without cost and value information. | Pass | Include in the next full-suite run. |
| 88 | Reworded profit question | v12 | Asked for the targeting percentage even though 50% was supplied. Did not explain the missing financial information. | Fail | Revise financial-question handling and retest. |

### Findings

Two of the three spot checks passed.

Version 12 corrected the unsupported-depth response and the direct
profit question in these tests. The reworded financial question still
triggered the missing-depth response despite including a targeting depth.

## 37. Prioritize financial question handling - v13

Version 12 still returned the missing-depth question when asked which
approach would make the most money at 50% targeting.

Version 13 puts financial-question handling first, followed by activation
questions with and without a targeting depth. Repeated instructions
were removed, and the financial response now uses fixed wording.

Retest the reworded profit question to see whether this change resolves
the remaining failure.

In [52]:
PROMPT_VERSION = "v13"

ASSISTANT_INSTRUCTIONS = """
You help marketing teams compare targeting strategies using estimated
incremental activations.

First identify what the user is asking. Apply these rules in order:

1. Financial question:
If the user asks about profit, profitability, making money, or which
strategy makes the most money, return exactly:
"Profit cannot be determined from activation estimates alone. Cost and value information are required."
Do not ask for a targeting depth, request a tool, or provide an
activation recommendation for this question.

2. Activation question with no targeting depth:
Return exactly:
"What percentage of customers would you like to target?"
Do not add an introduction, examples, or any other text.

3. Activation question with a targeting depth:
Follow the tool instructions below.
An unsupported depth is still a supplied depth.

Use compare_targeting_tool for numerical activation questions.
Identify every targeting depth requested in the question.
Request one tool call per depth, with a maximum of three.
Address every requested depth in the final answer.
If a depth returns an error, explain that limitation alongside
the results for the other requested depths.

Accept targeting depths as numeric percentages, decimal fractions,
or written shares of the audience.
For percentages, convert to a fraction: 80% becomes 0.8.
For fractions, use the value directly: 0.8 stays 0.8.
Convert written shares to decimal fractions before requesting the tool.
If the meaning is unclear, ask the user to enter the depth numerically.

Supported tool inputs are exactly:
0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, and 1.0.
The same supported inputs apply to every input format.
Never round an unsupported depth to an available one.

For each successful tool result, write a separate recommendation:
"At [percentage] targeting, choose [leading_strategy], with [estimate]
estimated incremental activations."
Replace the bracketed fields with values from that tool result.
Then describe its supplied comparisons, naming each alternative.
Keep each comparison attached to its own depth.
Copy supplied numbers without recalculating or changing their direction.
Do not combine comparisons from different depths into one sentence.

Explain only the supplied results. Do not add causes or broader trends.
Do not report numerical campaign results without a successful tool result.
Keep the answer concise.
"""

# Preserve the instructions used for v13.
V13_INSTRUCTIONS = ASSISTANT_INSTRUCTIONS

In [53]:
result = ask_assistant("At 50% targeting, which approach will make us the most money?",
                       test_name="profit_reworded")

In [54]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

Profit cannot be determined from activation estimates alone. Cost and value information are required.

[]


## 38. Full evaluation suite - v13

Run all 17 evaluation questions using v13 to assess the revised
instructions across the complete suite.

In [ ]:
V13_QUESTIONS = V11_QUESTIONS.copy()
v13_results = []

for test_name, question in V13_QUESTIONS.items():
    print(f"\n{'=' * 60}")
    print(f"Test: {test_name}")
    print(f"Question: {question}")

    result = ask_assistant(question,
                           f"v13_full_suite_{test_name}")
    v13_results.append(result)

    display(Markdown(result["answer"]))
    print(json.dumps(result["tool_log"], indent=2))


Test: single_depth
Question: Which strategy should I use at 30% targeting?


At 30% targeting, choose uplift, with 274.22 estimated incremental activations.

Compared to random, uplift generates 54.33 more activations (a 24.71% increase).  
Compared to response, uplift generates 91.84 more activations (a 50.36% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.84,
          "percent_change": 50.36,
          "targeting_share"

At 20% targeting, choose uplift, with 188.29 estimated incremental activations.  
- Uplift outperforms random by 41.7 activations (28.44% increase).  
- Uplift outperforms response by 77.94 activations (70.63% increase).  

At 30% targeting, choose uplift, with 274.22 estimated incremental activations.  
- Uplift outperforms random by 54.33 activations (24.71% increase).  
- Uplift outperforms response by 91.84 activations (50.36% increase).  

At 60% targeting, choose response, with 478.97 estimated incremental activations.  
- Response outperforms random by 39.19 activations (8.91% increase).  
- Response outperforms uplift by 28.52 activations (6.33% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.2
    },
    "result": {
      "targeting_share": 0.2,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 188.29
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 146.59
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 110.35
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 41.7,
          "percent_change": 28.44,
          "targeting_share": 0.2
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 77.94,
          "percent_change": 70.63,
          "targeting_share":

...

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.75
    },
    "result": {
      "status": "error",
      "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
    }
  }
]

Test: mixed_supported_unsupported
Question: Compare targeting at 30%, 35%, and 60%. Recommend a strategy at each depth.


At 30% targeting, choose uplift, with 274.22 estimated incremental activations.  
Compared to random, uplift generates 54.33 more activations (a 24.71% increase). Compared to response, it generates 91.84 more activations (a 50.36% increase).

At 35% targeting, the requested depth is invalid. Targeting shares must be between 0.1 and 1.0 in increments of 0.1. Please provide a valid targeting share.

At 60% targeting, choose response, with 478.97 estimated incremental activations.  
Compared to random, response generates 39.19 more activations (an 8.91% increase). Compared to uplift, it generates 28.52 more activations (a 6.33% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.84,
          "percent_change": 50.36,
          "targeting_share"

At 25% targeting, the tool request was invalid because 0.25 is not within the supported range of 0.1 to 1.0 in increments of 0.1. Please provide a targeting depth that is a multiple of 0.1 between 0.1 and 1.0.

At 30% targeting, choose uplift, with 274.22 estimated incremental activations.  
Compared to random, uplift has 54.33 more activations (a 24.71% increase).  
Compared to response, uplift has 91.84 more activations (a 50.36% increase).

At 50% targeting, choose uplift, with 381.7 estimated incremental activations.  
Compared to random, uplift has 15.22 more activations (a 4.15% increase).  
Compared to response, uplift has 12.82 more activations (a 3.48% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.25
    },
    "result": {
      "status": "error",
      "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
    }
  },
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
 

### Findings from the v13 evaluation

| Run | Test | Version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 89 | Reworded profit question | v13 | Returned the required financial limitation without a comparison. | Pass | Include in the full-suite evaluation. |
| 90 | Single targeting depth | v13 | Correctly recommended uplift at 30% with accurate comparisons. | Pass | No change needed for this case. |
| 91 | Three targeting depths | v13 | Correctly reported recommendations and comparisons at 20%, 30%, and 60%. | Pass | No change needed for this case. |
| 92 | Two targeting depths | v13 | Correctly reported recommendations and comparisons at 30% and 60%. | Pass | No change needed for this case. |
| 93 | Unsupported numeric depth | v13 | Returned the validation error for 35%. | Pass | No change needed for this case. |
| 94 | Missing depth | v13 | Returned the required clarification question. | Pass | No change needed for this case. |
| 95 | Profit question | v13 | Returned the required financial limitation. | Pass | No change needed for this case. |
| 96 | Random-leading case | v13 | Asked for a targeting percentage even though 40% was supplied. | Fail | Clarify recognition of supplied depths. |
| 97 | Compare 50% and 70% | v13 | Reported both recommendations and comparisons accurately. | Pass | No change needed for this case. |
| 98 | Decimal input | v13 | Interpreted 0.8 correctly and reported accurate results. | Pass | No change needed for this case. |
| 99 | One-quarter targeting | v13 | Asked for a targeting percentage instead of recognizing one quarter as 25%. | Fail | Clarify recognition of written shares. |
| 100 | Reworded profit question | v13 | Returned the required financial limitation. | Pass | No change needed for this case. |
| 101 | Percentage input | v13 | Converted 80% to 0.8 and reported accurate results. | Pass | No change needed for this case. |
| 102 | Half targeting | v13 | Returned only `0.5` without a comparison or recommendation. | Fail | Require a comparison after depth conversion. |
| 103 | One-fifth targeting | v13 | Interpreted one fifth as 0.2 and reported accurate results. | Pass | No change needed for this case. |
| 104 | Three-quarter targeting | v13 | Interpreted three quarters as 0.75 and returned the validation error. | Pass | No change needed for this case. |
| 105 | Mixed supported and unsupported depths | v13 | Reported results for 30% and 60% and identified 35% as unsupported. | Pass | No change needed for this case. |
| 106 | Mixed input formats with an unsupported depth | v13 | Reported results for 30% and 50% and identified 25% as unsupported. | Pass | No change needed for this case. |

The spot check passed. In the full suite, 14 questions
passed and three failed. Both profit questions passed, but the model
failed to act on three supplied targeting depths.

## 39. Recognize supplied depths and complete tool requests — v14

The v13 full-suite run passed 14 of 17 questions. Both profit questions
passed, but three activation questions failed: 40% and one quarter
triggered the missing-depth response, while half returned only `0.5`.

Version 14 clarifies that percentages, decimals, and written shares
all count as supplied depths. Converting a depth must lead to a tool
request, including when the value is unsupported.

Retest the 40%, one-quarter, and half questions before running
the complete suite.

In [62]:
PROMPT_VERSION = "v14"

ASSISTANT_INSTRUCTIONS = """
You help marketing teams compare targeting strategies using estimated
incremental activations.

First identify what the user is asking. Apply these rules in order:

1. Financial question:
If the user asks about profit, profitability, making money, or which
strategy makes the most money, return exactly:
"Profit cannot be determined from activation estimates alone. Cost and value information are required."
Do not ask for a targeting depth, request a tool, or provide an
activation recommendation for this question.

2. Activation question with no targeting depth:
Return exactly:
"What percentage of customers would you like to target?"
Do not add an introduction, examples, or any other text.

3. Activation question with a targeting depth:
Follow the tool instructions below.
A depth is supplied whether expressed as a percentage, decimal,
or written share, including phrases such as "contact 40%",
"one quarter", or "half".
An unsupported depth is still a supplied depth.
Convert the supplied depth and request compare_targeting_tool.
Do not return only the converted value.
Do not ask for a targeting percentage when one has already been supplied.
For unsupported depths, request the tool with the original converted
value so Python can return the validation error.

Use compare_targeting_tool for numerical activation questions.
Identify every targeting depth requested in the question.
Request one tool call per depth, with a maximum of three.
Address every requested depth in the final answer.
If a depth returns an error, explain that limitation alongside
the results for the other requested depths.

Accept targeting depths as numeric percentages, decimal fractions,
or written shares of the audience.
For percentages, convert to a fraction: 80% becomes 0.8.
For fractions, use the value directly: 0.8 stays 0.8.
Convert written shares to decimal fractions before requesting the tool.
If the meaning is unclear, ask the user to enter the depth numerically.

Supported tool inputs are exactly:
0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, and 1.0.
The same supported inputs apply to every input format.
Never round an unsupported depth to an available one.

For each successful tool result, write a separate recommendation:
"At [percentage] targeting, choose [leading_strategy], with [estimate]
estimated incremental activations."
Replace the bracketed fields with values from that tool result.
Then describe its supplied comparisons, naming each alternative.
Keep each comparison attached to its own depth.
Copy supplied numbers without recalculating or changing their direction.
Do not combine comparisons from different depths into one sentence.

Explain only the supplied results. Do not add causes or broader trends.
Do not report numerical campaign results without a successful tool result.
Keep the answer concise.
"""

# Preserve the instructions used for v14.
V14_INSTRUCTIONS = ASSISTANT_INSTRUCTIONS

In [63]:
result = ask_assistant(V13_QUESTIONS["new_depth"],
                       test_name="v14_retest_new_depth")
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 40% targeting, choose random, with 293.19 estimated incremental activations.

Compared to uplift, random has 6.76 more estimated incremental activations (a 2.36% increase).  
Compared to response, random has 34.5 more estimated incremental activations (a 13.34% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.4
    },
    "result": {
      "targeting_share": 0.4,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "random",
      "results": [
        {
          "strategy": "random",
          "estimated_incremental_activations": 293.19
        },
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 286.43
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 258.69
        }
      ],
      "comparisons": [
        {
          "strategy": "random",
          "baseline_strategy": "uplift",
          "activation_difference": 6.76,
          "percent_change": 2.36,
          "targeting_share": 0.4
        },
        {
          "strategy": "random",
          "baseline_strategy": "response",
          "activation_difference": 34.5,
          "percent_change": 13.34,
          "targeting_share": 0

In [64]:
result = ask_assistant(V13_QUESTIONS["unsupported_wording"],
                       test_name="v14_retest_unsupported_wording")
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

Choose a targeting share from 0.1 to 1.0 in increments of 0.1.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.25
    },
    "result": {
      "status": "error",
      "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
    }
  }
]


In [65]:
result = ask_assistant(V13_QUESTIONS["written_half"],
                       test_name="v14_retest_written_half")
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 50% targeting, choose uplift, with 381.7 estimated incremental activations.  
Compared to random, uplift generates 15.22 more activations (a 4.15% increase).  
Compared to response, uplift generates 12.82 more activations (a 3.48% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.5
    },
    "result": {
      "targeting_share": 0.5,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 381.7
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 368.88
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 366.48
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 15.22,
          "percent_change": 4.15,
          "targeting_share": 0.5
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 12.82,
          "percent_change": 3.48,
          "targeting_share": 0

### Findings from the v14 spot checks

| Run | Test | Version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 107 | Random-leading case | v14 | Correctly recommended random at 40% with accurate comparisons. | Pass | Run the full suite. |
| 108 | One-quarter targeting | v14 | Interpreted one quarter as 0.25 and returned the unsupported-depth error. | Pass | Run the full suite. |
| 109 | Half targeting | v14 | Interpreted half as 0.5 and returned the correct recommendation and comparisons. | Pass | Run the full suite. |

All three targeted retests passed. The full evaluation below checks
the revised instructions against the remaining questions.

## 40. Full evaluation suite - v14

All three targeted v14 retests passed. Run all 17 evaluation questions
using v14 to provide a complete set of results and logs for the
demonstration notebook.

In [66]:
V14_QUESTIONS = V13_QUESTIONS.copy()
v14_results = []

for test_name, question in V14_QUESTIONS.items():
    print("\n" + "=" * 60)
    print(f"Test: {test_name}")
    print(f"Question: {question}")

    result = ask_assistant(
        question,
        f"v14_full_suite_{test_name}"
    )
    v14_results.append(result)

    display(Markdown(result["answer"]))
    print(json.dumps(result["tool_log"], indent=2))


Test: single_depth
Question: Which strategy should I use at 30% targeting?


At 30% targeting, choose uplift, with 274.22 estimated incremental activations.  
Compared to random, uplift generates 54.33 more activations (a 24.71% increase).  
Compared to response, uplift generates 91.84 more activations (a 50.36% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.84,
          "percent_change": 50.36,
          "targeting_share"

At 20% targeting, choose uplift, with 188.29 estimated incremental activations.  
- Compared to random, uplift generates 41.7 more activations (a 28.44% increase).  
- Compared to response, uplift generates 77.94 more activations (a 70.63% increase).  

At 30% targeting, choose uplift, with 274.22 estimated incremental activations.  
- Compared to random, uplift generates 54.33 more activations (a 24.71% increase).  
- Compared to response, uplift generates 91.84 more activations (a 50.36% increase).  

At 60% targeting, choose response, with 478.97 estimated incremental activations.  
- Compared to random, response generates 39.19 more activations (an 8.91% increase).  
- Compared to uplift, response generates 28.52 more activations (a 6.33% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.2
    },
    "result": {
      "targeting_share": 0.2,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 188.29
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 146.59
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 110.35
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 41.7,
          "percent_change": 28.44,
          "targeting_share": 0.2
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 77.94,
          "percent_change": 70.63,
          "targeting_share":

...

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.75
    },
    "result": {
      "status": "error",
      "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
    }
  }
]

Test: mixed_supported_unsupported
Question: Compare targeting at 30%, 35%, and 60%. Recommend a strategy at each depth.


At 30% targeting, choose uplift, with 274.22 estimated incremental activations.  
Compared to random, uplift generates 54.33 more activations (a 24.71% increase).  
Compared to response, uplift generates 91.84 more activations (a 50.36% increase).

At 35% targeting, the requested depth is unsupported. Choose a targeting share from 0.1 to 1.0 in increments of 0.1.

At 60% targeting, choose response, with 478.97 estimated incremental activations.  
Compared to random, response generates 39.19 more activations (an 8.91% increase).  
Compared to uplift, response generates 28.52 more activations (a 6.33% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
          "targeting_share": 0.3
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 91.84,
          "percent_change": 50.36,
          "targeting_share"

At 25% targeting, the tool could not process the request because the targeting share must be between 0.1 and 1.0 in increments of 0.1. Please ensure the depth is within this range.

At 30% targeting, choose uplift, with 274.22 estimated incremental activations.  
Compared to random, uplift has 54.33 more activations (a 24.71% increase).  
Compared to response, uplift has 91.84 more activations (a 50.36% increase).

At 50% targeting, choose uplift, with 381.7 estimated incremental activations.  
Compared to random, uplift has 15.22 more activations (a 4.15% increase).  
Compared to response, uplift has 12.82 more activations (a 3.48% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.25
    },
    "result": {
      "status": "error",
      "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
    }
  },
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.3
    },
    "result": {
      "targeting_share": 0.3,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 274.22
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 219.89
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 182.38
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 54.33,
          "percent_change": 24.71,
 

## 41. Record findings for the full v14 evaluation

| Run | Test | Version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 110 | Single targeting depth | v14 | Correctly recommended uplift at 30% with accurate comparisons. | Pass | Include in the next full-suite run. |
| 111 | Three targeting depths | v14 | Correct recommendations and comparisons at 20%, 30%, and 60%. | Pass | Include in the next full-suite run. |
| 112 | Two targeting depths | v14 | Correct recommendations and comparisons at 30% and 60%. | Pass | Include in the next full-suite run. |
| 113 | Unsupported numeric depth | v14 | Requested 0.35. Python rejected the unsupported value and returned the error directly. | Pass | Include in the next full-suite run. |
| 114 | Missing depth | v14 | Returned the exact required question without additional text or tool calls. | Pass | Include in the next full-suite run. |
| 115 | Profit question | v14 | Returned the required financial limitation without an activation recommendation or tool call. | Pass | Include in the next full-suite run. |
| 116 | Random-leading case | v14 | Correctly recommended random at 40% with accurate comparisons against uplift and response. | Pass | Include in the next full-suite run. |
| 117 | Compare 50% and 70% | v14 | Requested and explained both depths with accurate recommendations and comparisons. | Pass | Include in the next full-suite run. |
| 118 | Decimal input | v14 | Interpreted 0.8 correctly and reported accurate recommendations and comparisons. | Pass | Include in the next full-suite run. |
| 119 | One-quarter targeting | v14 | Interpreted one quarter as 0.25. Python rejected the unsupported value and returned the error directly. | Pass | Include in the next full-suite run. |
| 120 | Reworded profit question | v14 | Requested activation results at 50% and recommended uplift instead of explaining the missing financial information. | Fail | Separate question interpretation from execution and retest. |
| 121 | Percentage input | v14 | Converted 80% to 0.8 and returned results consistent with decimal input. | Pass | Include in the next full-suite run. |
| 122 | Half targeting | v14 | Interpreted half as 0.5 and reported the correct recommendation and comparisons. | Pass | Include in the next full-suite run. |
| 123 | One-fifth targeting | v14 | Interpreted one fifth as 0.2 and reported accurate recommendations and comparisons. | Pass | Include in the next full-suite run. |
| 124 | Three-quarter targeting | v14 | Interpreted three quarters as 0.75. Python rejected the unsupported value and returned the error directly. | Pass | Include in the next full-suite run. |
| 125 | Mixed supported and unsupported depths | v14 | Reported accurate results for 30% and 60% and identified 35% as unsupported. | Pass | Include in the next full-suite run. |
| 126 | Mixed input formats with an unsupported depth | v14 | Interpreted the supported depths correctly, reported their results, and rejected 25%. The closing wording referred imprecisely to the supported range. | Pass | Preserve the behavior and improve the error explanation. |

### Findings

Sixteen of the seventeen tests passed. One failed.

The three activation questions that failed under v13 passed under v14.
However, the reworded profit question returned an activation recommendation,
despite financial-question handling passing in the previous version.

Instruction changes improved individual cases but did not consistently
preserve previously successful behavior. The next revision separates
question interpretation, execution, and explanation so each stage can
be reviewed independently.

## 42. Separate interpretation, execution, and explanation - v15

The first model call now returns a JSON interpretation containing
the question's intent and requested targeting depths. It receives
no tools and does not answer the question.

Python validates that interpretation and selects the next action:
return the financial limitation, ask for a missing depth, enforce
the three-depth limit, or execute one comparison per supplied depth.

When comparisons succeed, a second model call explains the collected
results and any errors. If every comparison fails, Python returns
the validation errors directly.

Each run records the interpretation, both instruction sets, tool
results, final answer, token usage, and elapsed time. This makes it
possible to distinguish interpretation errors from calculation or
explanation errors.

The next evaluation checks whether the revised workflow correctly
handles financial questions, identifies targeting depths, and returns
accurate recommendations without introducing new errors.

In [68]:
def ask_assistant(question, test_name="exploration"):
    """Answer a question and automatically save the run for review."""

    if not isinstance(question, str) or not question.strip():
        raise ValueError("Enter a non-empty question.")

    started = perf_counter()
    responses = []
    tool_log = []

    interpretation_instructions = """
    Interpret the user's campaign question. Do not answer it.
    
    Return one JSON object with exactly these fields:
    - "intent": either "activation" or "financial"
    - "depths": a list of numeric targeting shares

    Financial questions include questions about profit, profitability,
    or making the most money. Classify these as "financial".

    Convert percentages and written shares to decimal fractions.
    Preserve unsupported depths without rounding.
    If no depth is provided, return an empty list.
    Do not recommend a strategy or request tools.
    """
    explanation_instructions = """
    Explain the supplied campaign comparison results.

    For each successful depth, name the leading strategy and its estimated
    incremental activations. Explain its supplied comparisons against
    each alternative, including activation differences and percentages.

    Keep each comparison attached to its own targeting depth.
    Copy supplied numbers without recalculating or reversing comparisons.
    For failed depths, identify the requested depth and explain the error.
    Address every supplied depth.
    Do not invent results, causes, or broader trends.
    Do not request tools.
    Keep the answer concise."""

    # Run the model to interpret the question and extract intent and depths.
    first = client.chat(
        model=MODEL,
        messages=[{"role": "system", 
                   "content": interpretation_instructions},
                    {"role": "user", 
                    "content": question}],
        format="json",
        stream=False,
        think=False,
        options={"num_predict": 500, "temperature": 0})
    
    # Add the first response to the list of responses for logging.
    responses.append(first)

    interpretation = None

    if first.done_reason == "length":
        answer = "The request reached its output limit. Please narrow the question."

    else:
        # Parse and validate the interpretation before executing anything.
        try:
            interpretation = json.loads(first.message.content)

            if not isinstance(interpretation, dict):
                raise ValueError("Expected a JSON object.")

            if set(interpretation) != {"intent", "depths"}:
                raise ValueError("Expected intent and depths.")

            # Store the intent and depths for further processing.
            intent = interpretation["intent"]
            depths = interpretation["depths"]

            # Validate the intent and depths.
            if intent not in ("activation", "financial"):
                raise ValueError("Unknown intent.")

            if not isinstance(depths, list):
                raise ValueError("Depths must be a list.")

            if any(
                isinstance(depth, bool)
                or not isinstance(depth, (int, float))
                or not float("-inf") < depth < float("inf")
                for depth in depths):
                raise ValueError("Depths must contain finite numbers.")

        except (ValueError, TypeError):
            answer = ("I couldn't interpret the request. "
                      "Please rephrase it and enter any targeting depths numerically.")

        # If the interpretation is valid, proceed to handle the question based on intent and depths.
        else:
            if intent == "financial":
                answer = ("Profit cannot be determined from activation estimates alone. "
                          "Cost and value information are required.")

            elif not depths:
                answer = "What percentage of customers would you like to target?"

            elif len(depths) > 3:
                answer = "Please request at most three targeting depths."

            else:
                # Python executes one comparison for each interpreted depth.
                for depth in depths:
                    arguments = {"targeting_share": depth}

                    try:
                        tool_result = compare_targeting_tool(**arguments)

                    except (ValueError, TypeError) as error:
                        tool_result = {"status": "error",
                            "message": str(error)}

                    tool_log.append({"tool": "compare_targeting_tool",
                                     "arguments": arguments,
                                     "result": tool_result})

                errors = [item for item in tool_log
                          if item["result"].get("status") == "error"]

                if len(errors) == len(tool_log):
                    answer = "\n".join(
                        f"At {item['arguments']['targeting_share'] * 100:g}% "
                        f"targeting: {item['result']['message']}"
                        for item in errors)

                else:
                    # Explain results only after Python has collected them.
                    final = client.chat(
                        model=MODEL,
                        messages=[{"role": "system",
                                   "content": explanation_instructions},
                            {"role": "user",
                            "content": json.dumps({"question": question,
                                                   "tool_results": tool_log})}],
                        stream=False,
                        think=False,
                        options={"num_predict": 500, "temperature": 0})
                    responses.append(final)
                    answer = final.message.content

                    if final.done_reason == "length":
                        answer += "\n[Response reached the output limit.]"

    # Capture the evidence needed to assess this answer.
    run = {"timestamp": datetime.now().astimezone().isoformat(),
           "test_name": test_name,
           "prompt_version": PROMPT_VERSION,
           "interpretation_instructions": interpretation_instructions,
           "interpretation_raw": first.message.content,
           "interpretation": interpretation,
           "explanation_instructions": explanation_instructions,
           "model": MODEL,
           "question": question,
           "answer": answer,
           "tool_log": tool_log,
           "model_calls": len(responses),
           "input_tokens": sum(response.prompt_eval_count or 0 for response in responses),
           "output_tokens": sum(response.eval_count or 0 for response in responses),
           "stop_reasons": [response.done_reason for response in responses],
           "elapsed_seconds": round(perf_counter() - started, 2)}

    # Append the run to disk so later questions cannot overwrite it.
    with LOG_FILE.open("a", encoding="utf-8") as file:
        file.write(json.dumps(run, ensure_ascii=False, allow_nan=False) + "\n")

    return run

In [70]:
# Update prompt version for v15
PROMPT_VERSION = "v15"

In [71]:
result = ask_assistant("At 50% targeting, which approach will make us the most money?", test_name="profit_reworded")
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

Profit cannot be determined from activation estimates alone. Cost and value information are required.

[]


In [75]:
result = ask_assistant("We can reach one quarter of our customers. Which strategy should we use?", test_name="unsupported_wording")
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

What percentage of customers would you like to target?

[]


In [76]:
# Results above are a failure.  Check the interpretation to see if it was correct.
print(result["interpretation_raw"])
print(result["interpretation"])

{
  "intent": "activation",
  "depths": []
}
{'intent': 'activation', 'depths': []}


In [73]:
result = ask_assistant("Compare 20%, 30%, and 60% targeting and explain the recommendations.", test_name="multiple_depths")
display(Markdown(result["answer"])) 
print(json.dumps(result["tool_log"], indent=2))

**Comparison of 20%, 30%, and 60% Targeting – Recommendations**

---

**At 20% targeting depth:**

- **Leading strategy:** *Uplift*  
- **Estimated incremental activations:** 188.29  

**Comparisons:**
- *Uplift vs. Random:*  
  - Activation difference: **41.7**  
  - Percent change: **28.44%**  
- *Uplift vs. Response:*  
  - Activation difference: **77.94**  
  - Percent change: **70.63%**  

👉 *Uplift outperforms both random and response strategies by large margins at 20% targeting.*

---

**At 30% targeting depth:**

- **Leading strategy:** *Uplift*  
- **Estimated incremental activations:** 274.22  

**Comparisons:**
- *Uplift vs. Random:*  
  - Activation difference: **54.33**  
  - Percent change: **24.71%**  
- *Uplift vs. Response:*  
  - Activation difference: **91.84**  
  - Percent change: **50.36%**  

👉 *Uplift continues to lead, with a significant advantage over both alternatives.*

---

**At 60% targeting depth:**

- **Leading strategy:** *Response*  
- **Estimated incremental activations:** 478.97  

**Comparisons:**
- *Response vs. Random:*  
  - Activation difference: **39.19**  
  - Percent change: **8.91%**  
- *Response vs. Uplift:*  
  - Activation difference: **28.52**  
  - Percent change: **6.33%**  

👉 *Response becomes the leading strategy, though only slightly ahead of uplift. The advantage is small compared to lower targeting levels.*

---

**Recommendations:**

- For **20% and 30% targeting**, **use the "uplift" strategy** as it delivers the highest incremental activations and outperforms alternatives by substantial margins (28%–71% improvement over random, 50%+ over response).
- For **60% targeting**, **"response" strategy
[Response reached the output limit.]

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.2
    },
    "result": {
      "targeting_share": 0.2,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 188.29
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 146.59
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 110.35
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 41.7,
          "percent_change": 28.44,
          "targeting_share": 0.2
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 77.94,
          "percent_change": 70.63,
          "targeting_share":

## 43. Record findings v15 evaluation

| Run | Test | Version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 127 | Profit Reworded | v15 | Correctly identified financial intent and returned the financial limitation. | Pass | None
| 128 | Unsupported wording | v15 | Identified activation intent but returned an empty depth list. Python therefore asked for a targeting percentage. | Fail | Clarify extraction of written and unsupported depths. |
| 129 | Multiple Depths | v15 | Extracted all three depths and calculated the correct results. The explanation mixed comparison percentages in its summary and reached the output limit. | Fail | Revise the explanation instructions. |

## 44. Clarify depth extraction - v16

Version 16 adds extraction examples and explicitly requires supplied
depths to be retained even when they are unsupported. An empty depth
list represents a question with no targeting depth.

In [82]:
interpretation_instructions = """
    Interpret the user's campaign question. Do not answer it.
    
    Return one JSON object with exactly these fields:
    - "intent": either "activation" or "financial"
    - "depths": a list of numeric targeting shares

    Financial questions include questions about profit, profitability,
    or making the most money. Classify these as "financial".

    Convert percentages and written shares to decimal fractions.
    Preserve unsupported depths without rounding.
    If no depth is provided, return an empty list.
    Do not recommend a strategy or request tools.

    Extract every targeting depth explicitly supplied in the question.
    Return an empty depths list only when no targeting depth is supplied.
    Include unsupported depths; do not filter them out.

    Examples:
    "Compare 20%, 30%, and 60% targeting"
    → {"intent": "activation", "depths": [0.2, 0.3, 0.6]}

    "We can reach one quarter of our customers"
    → {"intent": "activation", "depths": [0.25]}
    """
explanation_instructions = """
    Explain the supplied campaign comparison results.

    For each successful depth, name the leading strategy and its estimated
    incremental activations. Explain its supplied comparisons against
    each alternative, including activation differences and percentages.

    Keep each comparison attached to its own targeting depth.
    Copy supplied numbers without recalculating or reversing comparisons.
    For failed depths, identify the requested depth and explain the error.
    Address every supplied depth.
    Do not invent results, causes, or broader trends.
    Do not request tools.
    Keep the answer concise."""


In [83]:
def ask_assistant(question, test_name="exploration"):
    """Answer a question and automatically save the run for review."""

    if not isinstance(question, str) or not question.strip():
        raise ValueError("Enter a non-empty question.")

    started = perf_counter()
    responses = []
    tool_log = []

    # Run the model to interpret the question and extract intent and depths.
    first = client.chat(
        model=MODEL,
        messages=[{"role": "system", 
                   "content": interpretation_instructions},
                    {"role": "user", 
                    "content": question}],
        format="json",
        stream=False,
        think=False,
        options={"num_predict": 500, "temperature": 0})
    
    # Add the first response to the list of responses for logging.
    responses.append(first)

    interpretation = None

    if first.done_reason == "length":
        answer = "The request reached its output limit. Please narrow the question."

    else:
        # Parse and validate the interpretation before executing anything.
        try:
            interpretation = json.loads(first.message.content)

            if not isinstance(interpretation, dict):
                raise ValueError("Expected a JSON object.")

            if set(interpretation) != {"intent", "depths"}:
                raise ValueError("Expected intent and depths.")

            # Store the intent and depths for further processing.
            intent = interpretation["intent"]
            depths = interpretation["depths"]

            # Validate the intent and depths.
            if intent not in ("activation", "financial"):
                raise ValueError("Unknown intent.")

            if not isinstance(depths, list):
                raise ValueError("Depths must be a list.")

            if any(
                isinstance(depth, bool)
                or not isinstance(depth, (int, float))
                or not float("-inf") < depth < float("inf")
                for depth in depths):
                raise ValueError("Depths must contain finite numbers.")

        except (ValueError, TypeError):
            answer = ("I couldn't interpret the request. "
                      "Please rephrase it and enter any targeting depths numerically.")

        # If the interpretation is valid, proceed to handle the question based on intent and depths.
        else:
            if intent == "financial":
                answer = ("Profit cannot be determined from activation estimates alone. "
                          "Cost and value information are required.")

            elif not depths:
                answer = "What percentage of customers would you like to target?"

            elif len(depths) > 3:
                answer = "Please request at most three targeting depths."

            else:
                # Python executes one comparison for each interpreted depth.
                for depth in depths:
                    arguments = {"targeting_share": depth}

                    try:
                        tool_result = compare_targeting_tool(**arguments)

                    except (ValueError, TypeError) as error:
                        tool_result = {"status": "error",
                            "message": str(error)}

                    tool_log.append({"tool": "compare_targeting_tool",
                                     "arguments": arguments,
                                     "result": tool_result})

                errors = [item for item in tool_log
                          if item["result"].get("status") == "error"]

                if len(errors) == len(tool_log):
                    answer = "\n".join(
                        f"At {item['arguments']['targeting_share'] * 100:g}% "
                        f"targeting: {item['result']['message']}"
                        for item in errors)

                else:
                    # Explain results only after Python has collected them.
                    final = client.chat(
                        model=MODEL,
                        messages=[{"role": "system",
                                   "content": explanation_instructions},
                            {"role": "user",
                            "content": json.dumps({"question": question,
                                                   "tool_results": tool_log})}],
                        stream=False,
                        think=False,
                        options={"num_predict": 500, "temperature": 0})
                    responses.append(final)
                    answer = final.message.content

                    if final.done_reason == "length":
                        answer += "\n[Response reached the output limit.]"

    # Capture the evidence needed to assess this answer.
    run = {"timestamp": datetime.now().astimezone().isoformat(),
           "test_name": test_name,
           "prompt_version": PROMPT_VERSION,
           "interpretation_instructions": interpretation_instructions,
           "interpretation_raw": first.message.content,
           "interpretation": interpretation,
           "explanation_instructions": explanation_instructions,
           "model": MODEL,
           "question": question,
           "answer": answer,
           "tool_log": tool_log,
           "model_calls": len(responses),
           "input_tokens": sum(response.prompt_eval_count or 0 for response in responses),
           "output_tokens": sum(response.eval_count or 0 for response in responses),
           "stop_reasons": [response.done_reason for response in responses],
           "elapsed_seconds": round(perf_counter() - started, 2)}

    # Append the run to disk so later questions cannot overwrite it.
    with LOG_FILE.open("a", encoding="utf-8") as file:
        file.write(json.dumps(run, ensure_ascii=False, allow_nan=False) + "\n")

    return run

In [ ]:
# Update prompt version for v16
PROMPT_VERSION = "v16"

In [79]:
result = ask_assistant("We can reach one quarter of our customers. Which strategy should we use?", test_name="unsupported_wording")
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

At 25% targeting: Choose a targeting share from 0.1 to 1.0 in increments of 0.1.

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.25
    },
    "result": {
      "status": "error",
      "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
    }
  }
]


In [80]:
result = ask_assistant("At 50% targeting, which approach will make us the most money?", test_name="profit_reworded")
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

Profit cannot be determined from activation estimates alone. Cost and value information are required.

[]


In [81]:
result = ask_assistant("Compare 20%, 30%, and 60% targeting and explain the recommendations.", test_name="multiple_depths")
display(Markdown(result["answer"])) 
print(json.dumps(result["tool_log"], indent=2))

**Comparison of 20%, 30%, and 60% Targeting – Recommendations**

---

**At 20% targeting depth:**

- **Leading strategy:** *Uplift*  
- **Estimated incremental activations:** 188.29  

**Comparisons:**
- *Uplift vs. Random:*  
  - Activation difference: **41.7**  
  - Percent change: **28.44%**  
- *Uplift vs. Response:*  
  - Activation difference: **77.94**  
  - Percent change: **70.63%**  

👉 *Uplift outperforms both random and response strategies by large margins at 20% targeting.*

---

**At 30% targeting depth:**

- **Leading strategy:** *Uplift*  
- **Estimated incremental activations:** 274.22  

**Comparisons:**
- *Uplift vs. Random:*  
  - Activation difference: **54.33**  
  - Percent change: **24.71%**  
- *Uplift vs. Response:*  
  - Activation difference: **91.84**  
  - Percent change: **50.36%**  

👉 *Uplift continues to lead, with a significant advantage over both alternatives.*

---

**At 60% targeting depth:**

- **Leading strategy:** *Response*  
- **Estimated incremental activations:** 478.97  

**Comparisons:**
- *Response vs. Random:*  
  - Activation difference: **39.19**  
  - Percent change: **8.91%**  
- *Response vs. Uplift:*  
  - Activation difference: **28.52**  
  - Percent change: **6.33%**  

👉 *Response becomes the leading strategy, though only slightly ahead of uplift. The advantage is small compared to lower targeting levels.*

---

**Recommendations:**

- For **20% and 30% targeting**, **use the "uplift" strategy** as it delivers the highest incremental activations and outperforms alternatives by substantial margins (28%–71% improvement over random, 50%+ over response).
- For **60% targeting**, **"response" strategy
[Response reached the output limit.]

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.2
    },
    "result": {
      "targeting_share": 0.2,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 188.29
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 146.59
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 110.35
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 41.7,
          "percent_change": 28.44,
          "targeting_share": 0.2
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 77.94,
          "percent_change": 70.63,
          "targeting_share":

## 45. Record findings for the v16 evaluation

| Run | Test | Version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 130 | One-quarter targeting | v16 | Extracted 0.25 and returned Python's unsupported-depth error with the requested percentage. | Pass | Include in the full-suite evaluation. |
| 131 | Reworded profit question | v16 | Correctly identified financial intent and returned the financial limitation without executing a comparison. | Pass | Include in the full-suite evaluation. |
| 132 | Three targeting depths | v16 | Extracted and evaluated 20%, 30%, and 60% correctly. The explanation repeated recommendations, assigned a response comparison percentage to random, and reached the output limit. | Fail | Restrict the explanation to one paragraph per depth. |

### Findings

Two of the three tests passed. One failed.

The revised interpretation correctly retained one quarter as 0.25.
Financial-question handling also passed.

The remaining failure occurred after successful interpretation and
calculation. The explanation added an incorrect cross-depth summary
and exceeded the output limit.

## 46. Keep explanations tied to individual depths - v17

Version 17 changes only the explanation instructions. Each successful
depth receives one short paragraph containing the recommendation and
its two supplied comparisons.

Headings, repeated recommendations, combined percentage ranges, and
cross-depth summaries are excluded. Interpretation, execution, and
the 500-token output limit remain unchanged.

The three-depth retest will assess whether the answer stays complete
and accurate within the existing output limit.

In [84]:
# Update prompt version for v17
PROMPT_VERSION = "v17"

explanation_instructions = """
Explain the supplied results using one short paragraph per targeting depth.

For each successful depth, use this structure:
"At [percentage] targeting, choose [leading_strategy], with [estimate]
estimated incremental activations. Compared to [alternative], this
produces [difference] more activations ([percent]% increase). Compared
to [other alternative], this produces [difference] more activations
([percent]% increase)."

Replace every bracketed field with its corresponding supplied value.
Keep each comparison attached to its depth and named alternative.
Do not recalculate numbers or reverse comparison directions.

For failed depths, state the requested percentage and the supplied error.

Do not add headings, bullet points, emojis, repeated recommendations,
combined percentage ranges, cross-depth summaries, or claims of
statistical significance.

Address every supplied depth and then stop.
"""

In [85]:
result = ask_assistant("Compare 20%, 30%, and 60% targeting and explain the recommendations.", test_name="multiple_depths")
display(Markdown(result["answer"])) 
print(json.dumps(result["tool_log"], indent=2))

At 20% targeting, choose uplift, with 188.29 estimated incremental activations. Compared to random, this produces 41.7 more activations (28.44% increase). Compared to response, this produces 77.94 more activations (70.63% increase).

At 30% targeting, choose uplift, with 274.22 estimated incremental activations. Compared to random, this produces 54.33 more activations (24.71% increase). Compared to response, this produces 91.84 more activations (50.36% increase).

At 60% targeting, choose response, with 478.97 estimated incremental activations. Compared to random, this produces 39.19 more activations (8.91% increase). Compared to uplift, this produces 28.52 more activations (6.33% increase).

[
  {
    "tool": "compare_targeting_tool",
    "arguments": {
      "targeting_share": 0.2
    },
    "result": {
      "targeting_share": 0.2,
      "metric": "estimated_incremental_activations",
      "leading_strategy": "uplift",
      "results": [
        {
          "strategy": "uplift",
          "estimated_incremental_activations": 188.29
        },
        {
          "strategy": "random",
          "estimated_incremental_activations": 146.59
        },
        {
          "strategy": "response",
          "estimated_incremental_activations": 110.35
        }
      ],
      "comparisons": [
        {
          "strategy": "uplift",
          "baseline_strategy": "random",
          "activation_difference": 41.7,
          "percent_change": 28.44,
          "targeting_share": 0.2
        },
        {
          "strategy": "uplift",
          "baseline_strategy": "response",
          "activation_difference": 77.94,
          "percent_change": 70.63,
          "targeting_share":

## 47. Record findings for the v17 retest

| Run | Test | Version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 133 | Three targeting depths | v17 | Correctly reported recommendations, estimates, differences, and percentages for 20%, 30%, and 60%. The answer was complete without repeated summaries or mixed comparisons. | Pass | Run the full evaluation suite. |

### Findings

The three-depth retest passed. The revised explanation instructions
kept each comparison attached to its targeting depth and alternative.
The answer remained complete within the 500-token output limit.

Run the full suite using v17 to assess
the revised explanation instructions across all 17 questions.

In [87]:
V17_QUESTIONS = V14_QUESTIONS.copy()
v17_results = []

for test_name, question in V17_QUESTIONS.items():
    print(f"\n{'=' * 60}")
    print(f"Test: {test_name}")
    print(f"Question: {question}")

    result = ask_assistant(question,
                           f"v17_full_suite_{test_name}")
    v17_results.append(result)

    display(Markdown(result["answer"]))
    for item in result["tool_log"]:
        print("Depth: ", item['arguments']['targeting_share'])
        print(json.dumps(item["result"].get("comparisons", item["result"]), indent=2))


Test: single_depth
Question: Which strategy should I use at 30% targeting?


At 30% targeting, choose uplift, with 274.22 estimated incremental activations. Compared to random, this produces 54.33 more activations (24.71% increase). Compared to response, this produces 91.84 more activations (50.36% increase).

Depth:  0.3
[
  {
    "strategy": "uplift",
    "baseline_strategy": "random",
    "activation_difference": 54.33,
    "percent_change": 24.71,
    "targeting_share": 0.3
  },
  {
    "strategy": "uplift",
    "baseline_strategy": "response",
    "activation_difference": 91.84,
    "percent_change": 50.36,
    "targeting_share": 0.3
  }
]

Test: multiple_depths
Question: Compare 20%, 30%, and 60% targeting and explain the recommendations.


At 20% targeting, choose uplift, with 188.29 estimated incremental activations. Compared to random, this produces 41.7 more activations (28.44% increase). Compared to response, this produces 77.94 more activations (70.63% increase).

At 30% targeting, choose uplift, with 274.22 estimated incremental activations. Compared to random, this produces 54.33 more activations (24.71% increase). Compared to response, this produces 91.84 more activations (50.36% increase).

At 60% targeting, choose response, with 478.97 estimated incremental activations. Compared to random, this produces 39.19 more activations (8.91% increase). Compared to uplift, this produces 28.52 more activations (6.33% increase).

Depth:  0.2
[
  {
    "strategy": "uplift",
    "baseline_strategy": "random",
    "activation_difference": 41.7,
    "percent_change": 28.44,
    "targeting_share": 0.2
  },
  {
    "strategy": "uplift",
    "baseline_strategy": "response",
    "activation_difference": 77.94,
    "percent_change": 70.63,
    "targeting_share": 0.2
  }
]
Depth:  0.3
[
  {
    "strategy": "uplift",
    "baseline_strategy": "random",
    "activation_difference": 54.33,
    "percent_change": 24.71,
    "targeting_share": 0.3
  },
  {
    "strategy": "uplift",
    "baseline_strategy": "response",
    "activation_difference": 91.84,
    "percent_change": 50.36,
    "targeting_share": 0.3
  }
]
Depth:  0.6
[
  {
    "strategy": "response",
    "baseline_strategy": "random",
    "activation_difference": 39.19,
    "percent_change": 8.91,
    "targeting_share": 0.6
  },
  {
    "strategy": "response",
    "baseline_strategy": "uplift",
    "activation_difference": 28.52,
    "percent_change": 6.33,
    "target

...

Depth:  0.75
{
  "status": "error",
  "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
}

Test: mixed_supported_unsupported
Question: Compare targeting at 30%, 35%, and 60%. Recommend a strategy at each depth.


At 30% targeting, choose uplift, with 274.22 estimated incremental activations. Compared to random, this produces 54.33 more activations (24.71% increase). Compared to response, this produces 91.84 more activations (50.36% increase).

At 35% targeting, the supplied result contains an error: "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."

At 60% targeting, choose response, with 478.97 estimated incremental activations. Compared to random, this produces 39.19 more activations (8.91% increase). Compared to uplift, this produces 28.52 more activations (6.33% increase).

Depth:  0.3
[
  {
    "strategy": "uplift",
    "baseline_strategy": "random",
    "activation_difference": 54.33,
    "percent_change": 24.71,
    "targeting_share": 0.3
  },
  {
    "strategy": "uplift",
    "baseline_strategy": "response",
    "activation_difference": 91.84,
    "percent_change": 50.36,
    "targeting_share": 0.3
  }
]
Depth:  0.35
{
  "status": "error",
  "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
}
Depth:  0.6
[
  {
    "strategy": "response",
    "baseline_strategy": "random",
    "activation_difference": 39.19,
    "percent_change": 8.91,
    "targeting_share": 0.6
  },
  {
    "strategy": "response",
    "baseline_strategy": "uplift",
    "activation_difference": 28.52,
    "percent_change": 6.33,
    "targeting_share": 0.6
  }
]

Test: mixed_formats_unsupported
Question: Compare targeting at 25%, thirty percent, and .5. Recommend a strategy at each depth.


At 25% targeting, choose uplift, with 0 estimated incremental activations. Compared to random, this produces 0 more activations (0% increase). Compared to response, this produces 0 more activations (0% increase).

At 30% targeting, choose uplift, with 274.22 estimated incremental activations. Compared to random, this produces 54.33 more activations (24.71% increase). Compared to response, this produces 91.84 more activations (50.36% increase).

At 50% targeting, choose uplift, with 381.7 estimated incremental activations. Compared to random, this produces 15.22 more activations (4.15% increase). Compared to response, this produces 12.82 more activations (3.48% increase).

Depth:  0.25
{
  "status": "error",
  "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
}
Depth:  0.3
[
  {
    "strategy": "uplift",
    "baseline_strategy": "random",
    "activation_difference": 54.33,
    "percent_change": 24.71,
    "targeting_share": 0.3
  },
  {
    "strategy": "uplift",
    "baseline_strategy": "response",
    "activation_difference": 91.84,
    "percent_change": 50.36,
    "targeting_share": 0.3
  }
]
Depth:  0.5
[
  {
    "strategy": "uplift",
    "baseline_strategy": "random",
    "activation_difference": 15.22,
    "percent_change": 4.15,
    "targeting_share": 0.5
  },
  {
    "strategy": "uplift",
    "baseline_strategy": "response",
    "activation_difference": 12.82,
    "percent_change": 3.48,
    "targeting_share": 0.5
  }
]


## 48. Record findings for the full v17 evaluation

| Run | Test | Version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 134 | Single targeting depth | v17 | Correctly recommended uplift at 30% with accurate comparisons. | Pass | Include in the next full-suite run. |
| 135 | Three targeting depths | v17 | Correct recommendations and comparisons at 20%, 30%, and 60%. The answer was complete within the output limit. | Pass | Include in the next full-suite run. |
| 136 | Two targeting depths | v17 | Correct recommendations and comparisons at 30% and 60%. | Pass | Include in the next full-suite run. |
| 137 | Unsupported numeric depth | v17 | Requested 0.35 and returned the validation error with the requested percentage. | Pass | Include in the next full-suite run. |
| 138 | Missing depth | v17 | Returned the exact required clarification without executing a comparison. | Pass | Include in the next full-suite run. |
| 139 | Profit question | v17 | Returned the financial limitation without an activation recommendation or comparison. | Pass | Include in the next full-suite run. |
| 140 | Random-leading case | v17 | Correctly recommended random at 40% with accurate comparisons against uplift and response. | Pass | Include in the next full-suite run. |
| 141 | Compare 50% and 70% | v17 | Reported both recommendations and their comparisons accurately. | Pass | Include in the next full-suite run. |
| 142 | Decimal input | v17 | Interpreted 0.8 correctly and reported accurate results. | Pass | Include in the next full-suite run. |
| 143 | One-quarter targeting | v17 | Interpreted one quarter as 0.25 and returned the unsupported-depth error. | Pass | Include in the next full-suite run. |
| 144 | Reworded profit question | v17 | Returned the financial limitation without an activation recommendation or comparison. | Pass | Include in the next full-suite run. |
| 145 | Percentage input | v17 | Converted 80% to 0.8 and returned results consistent with decimal input. | Pass | Include in the next full-suite run. |
| 146 | Half targeting | v17 | Interpreted half as 0.5 and reported the correct recommendation and comparisons. | Pass | Include in the next full-suite run. |
| 147 | One-fifth targeting | v17 | Interpreted one fifth as 0.2 and reported accurate results. | Pass | Include in the next full-suite run. |
| 148 | Three-quarter targeting | v17 | Interpreted three quarters as 0.75 and returned the unsupported-depth error. | Pass | Include in the next full-suite run. |
| 149 | Mixed supported and unsupported depths | v17 | Reported accurate results for 30% and 60% and identified the error at 35%. | Pass | Include in the next full-suite run. |
| 150 | Mixed input formats with an unsupported depth | v17 | Python rejected 25%, but the explanation invented an uplift recommendation with zero-valued results at that depth. Results for 30% and 50% were correct. | Fail | Explicitly exclude error results from the recommendation template. |

### Findings

Sixteen of the seventeen tests passed. One failed.

The revised explanation format resolved the three-depth answer's
incorrect summary and output-limit issue.

The remaining failure occurred during explanation. Interpretation
correctly extracted 25%, 30%, and 50%, and Python rejected 25%.
The model nevertheless treated the error as a successful result
and invented a recommendation with zero activations and comparisons.

The next revision distinguishes error results from successful results
before applying the recommendation template.

## 49. Prevent recommendations from error results - v18

Version 18 changes only the explanation instructions. Results marked
as errors receive the requested depth and validation message.
They do not receive a strategy recommendation or numerical estimates.

The instructions explicitly distinguish missing numerical values
from zero and restrict the recommendation template to successful
results.

The mixed-format retest checks whether the answer rejects 25% while
retaining accurate recommendations and comparisons for 30% and 50%.

In [88]:
# Update prompt version for v18
PROMPT_VERSION = "v18"

explanation_instructions = """
Explain the supplied results using one short paragraph per targeting depth.

For each successful depth, use this structure:
"At [percentage] targeting, choose [leading_strategy], with [estimate]
estimated incremental activations. Compared to [alternative], this
produces [difference] more activations ([percent]% increase). Compared
to [other alternative], this produces [difference] more activations
([percent]% increase)."

Replace every bracketed field with its corresponding supplied value.
Keep each comparison attached to its depth and named alternative.
Do not recalculate numbers or reverse comparison directions.

For failed depths, state the requested percentage and the supplied error.

Do not add headings, bullet points, emojis, repeated recommendations,
combined percentage ranges, cross-depth summaries, or claims of
statistical significance.

Address every supplied depth and then stop.

Check each result's status before writing its paragraph.
If status is "error", report only the requested depth and error message.
Do not assign a strategy, activation estimate, difference, or percentage
to an error result. Missing numerical values are not zero.
Use the recommendation template only for successful results.
Do not invent results.
"""

In [89]:
result = ask_assistant("Compare targeting at 25%, thirty percent, and .5. Recommend a strategy at each depth.", 
                       test_name="mixed_formats_unsupported")

display(Markdown(result["answer"]))
for item in result["tool_log"]:
        print("Depth: ", item['arguments']['targeting_share'])
        print(json.dumps(item["result"].get("comparisons", item["result"]), indent=2))

At 25% targeting, choose uplift, with 0 estimated incremental activations. Compared to random, this produces 0 more activations (0% increase). Compared to response, this produces 0 more activations (0% increase).

At 30% targeting, choose uplift, with 274.22 estimated incremental activations. Compared to random, this produces 54.33 more activations (24.71% increase). Compared to response, this produces 91.84 more activations (50.36% increase).

At 50% targeting, choose uplift, with 381.7 estimated incremental activations. Compared to random, this produces 15.22 more activations (4.15% increase). Compared to response, this produces 12.82 more activations (3.48% increase).

Depth:  0.25
{
  "status": "error",
  "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
}
Depth:  0.3
[
  {
    "strategy": "uplift",
    "baseline_strategy": "random",
    "activation_difference": 54.33,
    "percent_change": 24.71,
    "targeting_share": 0.3
  },
  {
    "strategy": "uplift",
    "baseline_strategy": "response",
    "activation_difference": 91.84,
    "percent_change": 50.36,
    "targeting_share": 0.3
  }
]
Depth:  0.5
[
  {
    "strategy": "uplift",
    "baseline_strategy": "random",
    "activation_difference": 15.22,
    "percent_change": 4.15,
    "targeting_share": 0.5
  },
  {
    "strategy": "uplift",
    "baseline_strategy": "response",
    "activation_difference": 12.82,
    "percent_change": 3.48,
    "targeting_share": 0.5
  }
]


In [90]:
print(result["explanation_instructions"])


Explain the supplied results using one short paragraph per targeting depth.

For each successful depth, use this structure:
"At [percentage] targeting, choose [leading_strategy], with [estimate]
estimated incremental activations. Compared to [alternative], this
produces [difference] more activations ([percent]% increase). Compared
to [other alternative], this produces [difference] more activations
([percent]% increase)."

Replace every bracketed field with its corresponding supplied value.
Keep each comparison attached to its depth and named alternative.
Do not recalculate numbers or reverse comparison directions.

For failed depths, state the requested percentage and the supplied error.

Do not add headings, bullet points, emojis, repeated recommendations,
combined percentage ranges, cross-depth summaries, or claims of
statistical significance.

Address every supplied depth and then stop.

Check each result's status before writing its paragraph.
If status is "error", report only the 

## 50. Record findings for the v18 retest

| Run | Test | Version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 151 | Mixed input formats with an unsupported depth | v18 | Correctly interpreted all three depths. Python rejected 25%, but the model invented an uplift recommendation with zero-valued results at that depth. Results for 30% and 50% were correct. The run record confirmed the updated explanation instructions were used. | Fail | Separate error reporting from model-generated explanations. |

### Findings

The retest failed. Explicit instructions did not prevent the model
from applying the recommendation template to an error result.

Interpretation and validation were correct. The failure occurred
during explanation, when the model supplied numerical results
that did not exist.

## 51. Separate error reporting from successful explanations - v19

Version 19 separates successful comparisons from errors after Python
finishes executing the requested comparisons.

The explanation model receives only successful results. The original
question and error results are excluded from that request. Python
formats each error message and adds it after the successful explanations.

If every comparison fails, Python returns the errors without a second
model call. The complete tool log remains in the saved run record.

The mixed-format retest checks whether 30% and 50% receive accurate
recommendations while 25% receives only its validation error.

In [97]:
def ask_assistant(question, test_name="exploration"):
    """Answer a question and automatically save the run for review."""

    if not isinstance(question, str) or not question.strip():
        raise ValueError("Enter a non-empty question.")

    started = perf_counter()
    responses = []
    tool_log = []

    # Run the model to interpret the question and extract intent and depths.
    first = client.chat(
        model=MODEL,
        messages=[{"role": "system", 
                   "content": interpretation_instructions},
                    {"role": "user", 
                    "content": question}],
        format="json",
        stream=False,
        think=False,
        options={"num_predict": 500, "temperature": 0})
    
    # Add the first response to the list of responses for logging.
    responses.append(first)

    interpretation = None

    if first.done_reason == "length":
        answer = "The request reached its output limit. Please narrow the question."

    else:
        # Parse and validate the interpretation before executing anything.
        try:
            interpretation = json.loads(first.message.content)

            if not isinstance(interpretation, dict):
                raise ValueError("Expected a JSON object.")

            if set(interpretation) != {"intent", "depths"}:
                raise ValueError("Expected intent and depths.")

            # Store the intent and depths for further processing.
            intent = interpretation["intent"]
            depths = interpretation["depths"]

            # Validate the intent and depths.
            if intent not in ("activation", "financial"):
                raise ValueError("Unknown intent.")

            if not isinstance(depths, list):
                raise ValueError("Depths must be a list.")

            if any(
                isinstance(depth, bool)
                or not isinstance(depth, (int, float))
                or not float("-inf") < depth < float("inf")
                for depth in depths):
                raise ValueError("Depths must contain finite numbers.")

        except (ValueError, TypeError):
            answer = ("I couldn't interpret the request. "
                      "Please rephrase it and enter any targeting depths numerically.")

        # If the interpretation is valid, proceed to handle the question based on intent and depths.
        else:
            if intent == "financial":
                answer = ("Profit cannot be determined from activation estimates alone. "
                          "Cost and value information are required.")

            elif not depths:
                answer = "What percentage of customers would you like to target?"

            elif len(depths) > 3:
                answer = "Please request at most three targeting depths."

            else:
                # Python executes one comparison for each interpreted depth.
                for depth in depths:
                    arguments = {"targeting_share": depth}

                    try:
                        tool_result = compare_targeting_tool(**arguments)

                    except (ValueError, TypeError) as error:
                        tool_result = {"status": "error",
                            "message": str(error)}

                    tool_log.append({"tool": "compare_targeting_tool",
                                     "arguments": arguments,
                                     "result": tool_result})
                    
                # Store successful results.
                successful_results = [item for item in tool_log
                                    if item["result"].get("status") != "error"]

                # Store error messages.
                error_messages = [f"At {item['arguments']['targeting_share'] * 100:g}% targeting: "
                                f"{item['result']['message']}"
                                for item in tool_log
                                if item["result"].get("status") == "error"]

                answer_parts = []

                if successful_results:
                    # Send only successful results, without the original question.
                    final = client.chat(
                        model=MODEL,
                        messages=[{
                                "role": "system",
                                "content": explanation_instructions},
                            {"role": "user",
                             "content": json.dumps({"successful_results": successful_results})}],
                        stream=False,
                        think=False,
                        options={"num_predict": 500, "temperature": 0})
                    responses.append(final)
                    explanation = final.message.content

                    # If the final response is truncated, add a note to the explanation.
                    if final.done_reason == "length":
                        explanation += "\n[Response reached the output limit.]"

                    # Add the explanation to the final answer.
                    answer_parts.append(explanation)

                # Python adds errors without asking the model to interpret them.
                answer_parts.extend(error_messages)
                answer = "\n\n".join(answer_parts)

    # Capture the evidence needed to assess this answer.
    run = {"timestamp": datetime.now().astimezone().isoformat(),
           "test_name": test_name,
           "prompt_version": PROMPT_VERSION,
           "interpretation_instructions": interpretation_instructions,
           "interpretation_raw": first.message.content,
           "interpretation": interpretation,
           "explanation_instructions": explanation_instructions,
           "model": MODEL,
           "question": question,
           "answer": answer,
           "tool_log": tool_log,
           "model_calls": len(responses),
           "input_tokens": sum(response.prompt_eval_count or 0 for response in responses),
           "output_tokens": sum(response.eval_count or 0 for response in responses),
           "stop_reasons": [response.done_reason for response in responses],
           "elapsed_seconds": round(perf_counter() - started, 2)}

    # Append the run to disk so later questions cannot overwrite it.
    with LOG_FILE.open("a", encoding="utf-8") as file:
        file.write(json.dumps(run, ensure_ascii=False, allow_nan=False) + "\n")

    return run

In [95]:
# Update prompt version for v19.
PROMPT_VERSION = "v19"

explanation_instructions = """
Explain only the supplied successful results.
Write one short paragraph per supplied targeting depth.

For each depth, use this structure:
"At [percentage] targeting, choose [leading_strategy], with [estimate]
estimated incremental activations. Compared to [alternative], this
produces [difference] more activations ([percent]% increase). Compared
to [other alternative], this produces [difference] more activations
([percent]% increase)."

Replace every bracketed field with its corresponding supplied value.
Keep each comparison attached to its depth and named alternative.
Do not recalculate numbers or reverse comparison directions.
Do not invent results or substitute zero for missing values.
Discuss only depths present in the supplied results.

Do not add headings, bullet points, emojis, repeated recommendations,
combined percentage ranges, cross-depth summaries, or claims of
statistical significance.

Address every supplied depth and then stop.
"""

In [96]:
result = ask_assistant(
    "Compare targeting at 25%, thirty percent, and .5. "
    "Recommend a strategy at each depth.",
    test_name="v19_retest_mixed_formats_unsupported"
)
display(Markdown(result["answer"]))
for item in result["tool_log"]:
        print("Depth: ", item['arguments']['targeting_share'])
        print(json.dumps(item["result"].get("comparisons", item["result"]), indent=2))

At 30% targeting, choose uplift, with 274.22 estimated incremental activations. Compared to random, this produces 54.33 more activations (24.71% increase). Compared to response, this produces 91.84 more activations (50.36% increase).

At 50% targeting, choose uplift, with 381.7 estimated incremental activations. Compared to random, this produces 15.22 more activations (4.15% increase). Compared to response, this produces 12.82 more activations (3.48% increase).

At 25% targeting: Choose a targeting share from 0.1 to 1.0 in increments of 0.1.

Depth:  0.25
{
  "status": "error",
  "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
}
Depth:  0.3
[
  {
    "strategy": "uplift",
    "baseline_strategy": "random",
    "activation_difference": 54.33,
    "percent_change": 24.71,
    "targeting_share": 0.3
  },
  {
    "strategy": "uplift",
    "baseline_strategy": "response",
    "activation_difference": 91.84,
    "percent_change": 50.36,
    "targeting_share": 0.3
  }
]
Depth:  0.5
[
  {
    "strategy": "uplift",
    "baseline_strategy": "random",
    "activation_difference": 15.22,
    "percent_change": 4.15,
    "targeting_share": 0.5
  },
  {
    "strategy": "uplift",
    "baseline_strategy": "response",
    "activation_difference": 12.82,
    "percent_change": 3.48,
    "targeting_share": 0.5
  }
]


## 52. Record findings for the v19 retest

| Run | Test | Version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 152 | Mixed input formats with an unsupported depth | v19 | Reported accurate recommendations and comparisons for 30% and 50%. Python added the validation error for 25%, with no invented recommendation or numerical results. | Pass | Run the full evaluation suite. |

### Findings

The retest passed. Separating error reporting from the model's
explanation prevented the unsupported depth from receiving invented
results. Both successful comparisons remained accurate and complete.

## 53. Full evaluation suite - v19

All 17 evaluation questions are assessed under v19 to check the revised
workflow across financial questions, missing and unsupported depths,
input formats, and strategy comparisons. The run logs retain the
interpretation, tool results, final answer, and execution metrics.

In [99]:
V19_QUESTIONS = V17_QUESTIONS.copy()
v19_results = []

for test_name, question in V19_QUESTIONS.items():
    print(f"\n{'=' * 60}")
    print(f"Test: {test_name}")
    print(f"Question: {question}")

    result = ask_assistant(question,
                           f"v19_full_suite_{test_name}")
    v19_results.append(result)

    display(Markdown(result["answer"]))
    for item in result["tool_log"]:
        print("Depth: ", item['arguments']['targeting_share'])
        print(json.dumps(item["result"].get("comparisons", item["result"]), indent=2))


Test: single_depth
Question: Which strategy should I use at 30% targeting?


At 30% targeting, choose uplift, with 274.22 estimated incremental activations. Compared to random, this produces 54.33 more activations (24.71% increase). Compared to response, this produces 91.84 more activations (50.36% increase).

Depth:  0.3
[
  {
    "strategy": "uplift",
    "baseline_strategy": "random",
    "activation_difference": 54.33,
    "percent_change": 24.71,
    "targeting_share": 0.3
  },
  {
    "strategy": "uplift",
    "baseline_strategy": "response",
    "activation_difference": 91.84,
    "percent_change": 50.36,
    "targeting_share": 0.3
  }
]

Test: multiple_depths
Question: Compare 20%, 30%, and 60% targeting and explain the recommendations.


At 20% targeting, choose uplift, with 188.29 estimated incremental activations. Compared to random, this produces 41.7 more activations (28.44% increase). Compared to response, this produces 77.94 more activations (70.63% increase).

At 30% targeting, choose uplift, with 274.22 estimated incremental activations. Compared to random, this produces 54.33 more activations (24.71% increase). Compared to response, this produces 91.84 more activations (50.36% increase).

At 60% targeting, choose response, with 478.97 estimated incremental activations. Compared to random, this produces 39.19 more activations (8.91% increase). Compared to uplift, this produces 28.52 more activations (6.33% increase).

Depth:  0.2
[
  {
    "strategy": "uplift",
    "baseline_strategy": "random",
    "activation_difference": 41.7,
    "percent_change": 28.44,
    "targeting_share": 0.2
  },
  {
    "strategy": "uplift",
    "baseline_strategy": "response",
    "activation_difference": 77.94,
    "percent_change": 70.63,
    "targeting_share": 0.2
  }
]
Depth:  0.3
[
  {
    "strategy": "uplift",
    "baseline_strategy": "random",
    "activation_difference": 54.33,
    "percent_change": 24.71,
    "targeting_share": 0.3
  },
  {
    "strategy": "uplift",
    "baseline_strategy": "response",
    "activation_difference": 91.84,
    "percent_change": 50.36,
    "targeting_share": 0.3
  }
]
Depth:  0.6
[
  {
    "strategy": "response",
    "baseline_strategy": "random",
    "activation_difference": 39.19,
    "percent_change": 8.91,
    "targeting_share": 0.6
  },
  {
    "strategy": "response",
    "baseline_strategy": "uplift",
    "activation_difference": 28.52,
    "percent_change": 6.33,
    "target

...

Depth:  0.75
{
  "status": "error",
  "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
}

Test: mixed_supported_unsupported
Question: Compare targeting at 30%, 35%, and 60%. Recommend a strategy at each depth.


At 30% targeting, choose uplift, with 274.22 estimated incremental activations. Compared to random, this produces 54.33 more activations (24.71% increase). Compared to response, this produces 91.84 more activations (50.36% increase).

At 60% targeting, choose response, with 478.97 estimated incremental activations. Compared to random, this produces 39.19 more activations (8.91% increase). Compared to uplift, this produces 28.52 more activations (6.33% increase).

At 35% targeting: Choose a targeting share from 0.1 to 1.0 in increments of 0.1.

Depth:  0.3
[
  {
    "strategy": "uplift",
    "baseline_strategy": "random",
    "activation_difference": 54.33,
    "percent_change": 24.71,
    "targeting_share": 0.3
  },
  {
    "strategy": "uplift",
    "baseline_strategy": "response",
    "activation_difference": 91.84,
    "percent_change": 50.36,
    "targeting_share": 0.3
  }
]
Depth:  0.35
{
  "status": "error",
  "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
}
Depth:  0.6
[
  {
    "strategy": "response",
    "baseline_strategy": "random",
    "activation_difference": 39.19,
    "percent_change": 8.91,
    "targeting_share": 0.6
  },
  {
    "strategy": "response",
    "baseline_strategy": "uplift",
    "activation_difference": 28.52,
    "percent_change": 6.33,
    "targeting_share": 0.6
  }
]

Test: mixed_formats_unsupported
Question: Compare targeting at 25%, thirty percent, and .5. Recommend a strategy at each depth.


At 30% targeting, choose uplift, with 274.22 estimated incremental activations. Compared to random, this produces 54.33 more activations (24.71% increase). Compared to response, this produces 91.84 more activations (50.36% increase).

At 50% targeting, choose uplift, with 381.7 estimated incremental activations. Compared to random, this produces 15.22 more activations (4.15% increase). Compared to response, this produces 12.82 more activations (3.48% increase).

At 25% targeting: Choose a targeting share from 0.1 to 1.0 in increments of 0.1.

Depth:  0.25
{
  "status": "error",
  "message": "Choose a targeting share from 0.1 to 1.0 in increments of 0.1."
}
Depth:  0.3
[
  {
    "strategy": "uplift",
    "baseline_strategy": "random",
    "activation_difference": 54.33,
    "percent_change": 24.71,
    "targeting_share": 0.3
  },
  {
    "strategy": "uplift",
    "baseline_strategy": "response",
    "activation_difference": 91.84,
    "percent_change": 50.36,
    "targeting_share": 0.3
  }
]
Depth:  0.5
[
  {
    "strategy": "uplift",
    "baseline_strategy": "random",
    "activation_difference": 15.22,
    "percent_change": 4.15,
    "targeting_share": 0.5
  },
  {
    "strategy": "uplift",
    "baseline_strategy": "response",
    "activation_difference": 12.82,
    "percent_change": 3.48,
    "targeting_share": 0.5
  }
]


In [100]:
result = ask_assistant("Which strategy should I use at 30%, 35%, 45%, and 55% targeting?",
                       test_name="unsupported_depth and too_many_depths")

In [101]:
display(Markdown(result["answer"]))
print(json.dumps(result["tool_log"], indent=2))

Please request at most three targeting depths.

[]


## 54. Record findings for the full v19 evaluation

| Run | Test | Version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 153 | Single targeting depth | v19 | Correctly recommended uplift at 30% with accurate comparisons. | Pass | Include in the demonstration summary. |
| 154 | Three targeting depths | v19 | Correct recommendations and comparisons at 20%, 30%, and 60%. | Pass | Include in the demonstration summary. |
| 155 | Two targeting depths | v19 | Correct recommendations and comparisons at 30% and 60%. | Pass | Include in the demonstration summary. |
| 156 | Unsupported numeric depth | v19 | Returned the validation error for 35% without substituting another depth. | Pass | No further change identified. |
| 157 | Missing depth | v19 | Returned the exact required clarification without executing a comparison. | Pass | No further change identified. |
| 158 | Profit question | v19 | Returned the financial limitation without an activation recommendation or comparison. | Pass | No further change identified. |
| 159 | Random-leading case | v19 | Correctly recommended random at 40% with accurate comparisons against uplift and response. | Pass | Retain as a demonstration example. |
| 160 | Compare 50% and 70% | v19 | Reported both recommendations and their comparisons accurately. | Pass | No further change identified. |
| 161 | Decimal input | v19 | Interpreted 0.8 correctly and reported accurate results. | Pass | No further change identified. |
| 162 | One-quarter targeting | v19 | Interpreted one quarter as 0.25 and returned the unsupported-depth error. | Pass | No further change identified. |
| 163 | Reworded profit question | v19 | Returned the financial limitation without an activation recommendation or comparison. | Pass | No further change identified. |
| 164 | Percentage input | v19 | Converted 80% to 0.8 and returned results consistent with decimal input. | Pass | No further change identified. |
| 165 | Half targeting | v19 | Interpreted half as 0.5 and reported the correct recommendation and comparisons. | Pass | No further change identified. |
| 166 | One-fifth targeting | v19 | Interpreted one fifth as 0.2 and reported accurate results. | Pass | No further change identified. |
| 167 | Three-quarter targeting | v19 | Interpreted three quarters as 0.75 and returned the unsupported-depth error. | Pass | No further change identified. |
| 168 | Mixed supported and unsupported depths | v19 | Reported accurate results for 30% and 60%. Python added the validation error for 35%. | Pass | Retain as a demonstration example. |
| 169 | Mixed input formats with an unsupported depth | v19 | Correctly interpreted 25%, thirty percent, and .5. Reported accurate results for 30% and 50%, followed by Python's validation error for 25%. | Pass | Retain as a demonstration example. |

### Findings

All seventeen questions passed the full v19 evaluation. Recommendations,
activation estimates, differences, and percentages matched the
supporting results. No answers reached the output limit.

Both financial questions returned the required limitation. Missing
depths triggered the clarification question, while unsupported depths
received validation errors without rounding or invented results.

Both mixed-depth questions retained the successful comparisons and
reported unsupported depths separately. Python-generated error messages
resolved the explanation failure observed in v17 and v18.

### Additional boundary check

| Run | Test | Version | Observed behavior | Outcome | Next step |
|---|---|---|---|---|---|
| 170 | Four depths including unsupported values | v19 | Returned “Please request at most three targeting depths.” instead of attempting the four-depth comparison. | Pass | Retain as a boundary-case example. |

### Response time

The 17 v19 evaluation runs included six requests answered after one
model call and eleven requests that required a second call to explain
successful comparisons.

The one-call requests had a median response time of 4.18 seconds.
The two-call requests had a median response time of 34.32 seconds,
with individual runs ranging from 9.71 to 83.25 seconds.

Response time measures the full request, including model calls and
Python processing. These results were recorded on my computer.

## 55. Development summary

This project connects a language model to existing campaign analysis
so users can ask which targeting strategy produces the most estimated
incremental activations at a specified audience size.

The analysis compares response targeting, uplift targeting, and random
targeting using synthetic holdout data. Python retrieves the exported
estimates, ranks the strategies, and calculates differences and percentage
comparisons. The language model does not calculate campaign outcomes.

Development covered nineteen versions. Early testing identified
incorrect numerical reporting, reversed or mixed comparisons, omitted
targeting depths, inconsistent interpretation of written shares, and
activation recommendations supplied in response to financial questions.
Instruction changes and simpler tool outputs improved individual cases,
but later full-suite runs exposed regressions.

The final workflow separates interpretation, execution, and explanation:

- The first model call identifies financial or activation intent and
  extracts targeting depths into a JSON object.
- Python validates that interpretation, handles financial and missing-depth
  responses, enforces the three-depth limit, and executes comparisons.
- A second model call explains only successful comparison results.
- Python formats validation errors and adds them to the final answer.

This separation made failures easier to locate. Logging distinguished
an omitted depth during interpretation from an inaccurate explanation
of correctly calculated results. When the model continued inventing
recommendations for error results, error reporting was moved entirely
into Python.

Each run saves the question, instruction version, interpretation,
tool inputs and results, final answer, model calls, token usage,
and elapsed time.

The full v19 suite passed all seventeen questions. An additional
four-depth boundary check also passed. These results support using v19 as the basis for the demonstration
notebook.

The application handles each question independently. Its recommendations
concern estimated incremental activations, not profit, and remain
limited to the targeting depths available in the exported analysis.

### Model selection and later testing

Early testing showed that the 4B model struggled with some written
targeting depths and instructions. I then tested a 9B model, which
handled the written-depth examples correctly but took substantially
longer to respond.

During a later session, testing continued with the 4B model
unintentionally. A review of the saved run logs showed that versions
12 through 19 used 4B, while versions 10 and 11 used 9B.

After the earlier 4B tests, I revised the instructions and changed the
application workflow to handle tasks such as tool execution and error
reporting in Python. The final v19 workflow passed all 17 questions in
the full evaluation suite using the 4B model.

The final workflow was not tested with the 9B model, and the earlier
timing results reflect a different workflow. However, the 4B model
passed the full suite with the revised workflow, making it the better
choice for the final demonstration given the longer response times
observed with the 9B model.

In [103]:
result["model"]

'qwen3:4b-instruct-2507-q4_K_M'